In [ ]:
"""
ICU AI Assistant Dashboard — Light Blue Clinical Theme
8-tab clinical decision support interface
Run: streamlit run ICUDashboard_LightBlue.py
"""
import streamlit as st
import pandas as pd
import numpy as np
import json
import base64
import plotly.graph_objects as go
from pathlib import Path
import datetime
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════
# CREDENTIALS (simple demo — replace with real auth)
# ══════════════════════════════════════════════════
VALID_CREDENTIALS = {
    "DR001": "icu2024",
    "DR002": "icu2024",
    "DR003": "icu2024",
    "ADMIN": "admin123",
}

# ══════════════════════════════════════════════════
# SESSION STATE INIT
# ══════════════════════════════════════════════════
if 'logged_in' not in st.session_state:
    st.session_state['logged_in']   = False
    st.session_state['doctor_id']   = ''
    st.session_state['login_error'] = ''


# ══════════════════════════════════════════════════
# PAGE CONFIG
# ══════════════════════════════════════════════════
st.set_page_config(
    page_title="ICU AI Assistant",
    page_icon="🏥",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ══════════════════════════════════════════════════
# DESIGN SYSTEM — LIGHT BLUE THEME
# ══════════════════════════════════════════════════
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@300;400;500;700&family=Syne:wght@400;500;600;700;800&display=swap');

:root {
    --bg-void:    #EBF4FB;
    --bg-deep:    #DDF0F9;
    --bg-panel:   #D0E8F5;
    --bg-card:    #C4E0F0;
    --bg-hover:   #B5D8EC;
    --border:     #90BDD8;
    --border-mid: #6CA8CC;
    --text-prime: #0D1B2A;
    --text-mid:   #1E3448;
    --text-dim:   #3A5F7A;
    --accent-blue:#1565C0;
    --accent-cyan:#0097A7;
}
* { box-sizing: border-box; }
html, body, [class*="css"] {
    font-family: 'Syne', sans-serif;
    background: var(--bg-void) !important;
    color: var(--text-prime);
}
.stApp {
    background: var(--bg-void) !important;
    background-image:
        radial-gradient(ellipse at 10% 0%, rgba(21,101,192,0.10) 0%, transparent 55%),
        radial-gradient(ellipse at 90% 100%, rgba(0,151,167,0.08) 0%, transparent 55%),
        radial-gradient(ellipse at 50% 50%, rgba(208,232,245,0.8) 0%, transparent 100%);
}
section[data-testid="stSidebar"] {
    background: var(--bg-deep) !important;
    border-right: 1px solid var(--border) !important;
    width: 290px !important;
}
section[data-testid="stSidebar"] > div { padding: 0 !important; }
header[data-testid="stHeader"] { display: none; }
.block-container { padding: 1.5rem 2rem !important; max-width: 100% !important; }

.stTabs [data-baseweb="tab-list"] {
    background: var(--bg-panel); border-bottom: 1px solid var(--border);
    border-radius: 8px 8px 0 0; padding: 0 8px; gap: 2px;
}
.stTabs [data-baseweb="tab"] {
    font-family: 'JetBrains Mono', monospace; font-size: 11px; font-weight: 600;
    letter-spacing: 0.06em; text-transform: uppercase; color: var(--text-dim) !important;
    padding: 12px 18px; border-bottom: 3px solid transparent;
    background: transparent !important; border-radius: 6px 6px 0 0; transition: all 0.2s ease;
}
.stTabs [data-baseweb="tab"]:hover { color: var(--text-mid) !important; background: rgba(21,101,192,0.06) !important; }
.stTabs [aria-selected="true"] { color: #1565C0 !important; background: rgba(21,101,192,0.1) !important; border-bottom: 3px solid #1565C0 !important; }
.stTabs [data-baseweb="tab-panel"] {
    background: var(--bg-panel); border: 1px solid var(--border); border-top: none;
    border-radius: 0 0 8px 8px; padding: 20px;
}

.stSelectbox > div > div {
    background: var(--bg-card) !important; border: 1px solid var(--border-mid) !important;
    border-radius: 6px !important; color: var(--text-prime) !important;
    font-family: 'JetBrains Mono', monospace !important; font-size: 13px !important;
}
[data-baseweb="popover"] { background: #D0E8F5 !important; }
[data-baseweb="menu"] { background: #D0E8F5 !important; border: 1px solid var(--border-mid) !important; border-radius: 6px !important; }
[data-baseweb="option"] { background: #D0E8F5 !important; color: #0D1B2A !important; font-family: 'JetBrains Mono', monospace !important; font-size: 13px !important; }
[data-baseweb="option"]:hover, [data-baseweb="option"][aria-selected="true"] { background: #B5D8EC !important; color: #1565C0 !important; }
ul[role="listbox"] { background: #D0E8F5 !important; }
li[role="option"] { background: #D0E8F5 !important; color: #0D1B2A !important; font-family: 'JetBrains Mono', monospace !important; font-size: 13px !important; padding: 8px 14px !important; }
li[role="option"]:hover, li[role="option"][aria-selected="true"] { background: #B5D8EC !important; color: #1565C0 !important; }

.stTextInput > div > div > input {
    background: var(--bg-card) !important; border: 1px solid var(--border-mid) !important;
    border-radius: 6px !important; color: var(--text-prime) !important;
    font-family: 'JetBrains Mono', monospace !important; font-size: 13px !important;
}
.stTextInput > div > div > input:focus { border-color: var(--accent-blue) !important; box-shadow: 0 0 0 3px rgba(21,101,192,0.15) !important; }
.stMultiSelect > div > div { background: var(--bg-card) !important; border: 1px solid var(--border-mid) !important; border-radius: 6px !important; color: var(--text-prime) !important; }
.stMultiSelect span[data-baseweb="tag"] { background: rgba(21,101,192,0.2) !important; color: #1565C0 !important; border: 1px solid rgba(21,101,192,0.4) !important; }

div[data-testid="stMetric"] { background: var(--bg-card); border: 1px solid var(--border); border-radius: 8px; padding: 16px !important; }
div[data-testid="stMetricValue"] { font-family: 'JetBrains Mono', monospace; font-size: 28px !important; color: var(--text-prime) !important; }

.stButton > button {
    background: var(--bg-hover) !important; color: var(--text-prime) !important;
    border: 1px solid var(--border-mid) !important; border-radius: 6px !important;
    font-family: 'Syne', sans-serif !important; font-size: 13px !important;
    font-weight: 600 !important; padding: 8px 20px !important; transition: all 0.15s ease !important;
}
.stButton > button:hover { background: rgba(21,101,192,0.15) !important; border-color: var(--accent-blue) !important; color: #1565C0 !important; }
button[kind="primary"] { background: #1565C0 !important; border-color: #1565C0 !important; color: #FFFFFF !important; }
button[kind="primary"]:hover { background: #1976D2 !important; }

.stForm { background: var(--bg-card) !important; border: 1px solid var(--border) !important; border-radius: 8px !important; padding: 20px !important; }
.stRadio > div { gap: 8px !important; }
.stRadio label { color: var(--text-mid) !important; font-size: 13px !important; }
.stSlider > div > div > div { background: var(--accent-blue) !important; }

.stSelectbox label, .stTextInput label, .stMultiSelect label, .stRadio label > div, .stSlider label {
    color: var(--text-mid) !important; font-family: 'JetBrains Mono', monospace !important;
    font-size: 10px !important; font-weight: 600 !important; letter-spacing: 0.1em !important; text-transform: uppercase !important;
}
hr { border-color: var(--border) !important; margin: 12px 0 !important; }
::-webkit-scrollbar { width: 4px; height: 4px; }
::-webkit-scrollbar-track { background: var(--bg-void); }
::-webkit-scrollbar-thumb { background: var(--border-mid); border-radius: 2px; }
::-webkit-scrollbar-thumb:hover { background: var(--accent-blue); }

.stWarning { background: rgba(200,146,10,0.10) !important; border-left: 3px solid #C8920A !important; border-radius: 6px !important; }
.stInfo    { background: rgba(21,101,192,0.08) !important; border-left: 3px solid #1565C0 !important; border-radius: 6px !important; }
.stError   { background: rgba(198,40,40,0.08) !important; border-left: 3px solid #C62828 !important; border-radius: 6px !important; }
.stSuccess { background: rgba(27,136,57,0.08) !important; border-left: 3px solid #1B8839 !important; border-radius: 6px !important; }
.stDataFrame { border: 1px solid var(--border) !important; border-radius: 6px !important; }

/* HERO */
.hero-banner { position:relative; width:100%; height:210px; border-radius:12px; overflow:hidden; margin-bottom:20px; border:1px solid #90BDD8; box-shadow:0 4px 24px rgba(21,101,192,0.15); }
.hero-img { position:absolute; inset:0; width:100%; height:100%; object-fit:cover; object-position:right 30%; opacity:0.80; }
.hero-fallback { position:absolute; inset:0; background:linear-gradient(135deg,#DDF0F9 0%,#C4E0F0 50%,#B5D8EC 100%); }
.hero-overlay { position:absolute; inset:0; background:linear-gradient(to right,rgba(235,244,251,0.97) 0%,rgba(235,244,251,0.80) 45%,rgba(235,244,251,0.30) 100%); display:flex; align-items:center; padding:0 40px; }
.hero-title { font-family:'Syne',sans-serif; font-size:32px; font-weight:800; color:#0D1B2A; letter-spacing:-0.02em; line-height:1.1; text-shadow:0 1px 4px rgba(255,255,255,0.8); }
.hero-sub { font-family:'JetBrains Mono',monospace; font-size:11px; font-weight:500; color:#1565C0; letter-spacing:0.12em; text-transform:uppercase; margin-top:7px; }
.hero-stats { display:flex; align-items:center; gap:28px; margin-top:18px; flex-wrap:wrap; }
.hero-stat-val { font-family:'JetBrains Mono',monospace; font-size:24px; font-weight:700; color:#0D1B2A; line-height:1; }
.hero-stat-label { font-family:'JetBrains Mono',monospace; font-size:10px; color:#3A5F7A; text-transform:uppercase; letter-spacing:0.1em; margin-top:4px; }
.hero-divider { width:1px; height:36px; background:rgba(13,27,42,0.2); align-self:center; }

/* COMPONENTS */
.icu-section-label { font-family:'JetBrains Mono',monospace; font-size:11px; font-weight:700; letter-spacing:0.18em; text-transform:uppercase; color:#3A5F7A; margin:20px 0 12px 0; padding-bottom:6px; border-bottom:1px solid #90BDD8; }
.icu-kpi { background:var(--bg-card); border:1px solid var(--border); border-radius:8px; padding:16px 18px; position:relative; overflow:hidden; transition:border-color 0.2s; }
.icu-kpi:hover { border-color:var(--border-mid); }
.icu-kpi::before { content:''; position:absolute; top:0; left:0; width:3px; height:100%; background:var(--accent-color,#1565C0); border-radius:3px 0 0 3px; }
.icu-kpi-label { font-family:'JetBrains Mono',monospace; font-size:10px; font-weight:700; letter-spacing:0.12em; text-transform:uppercase; color:#3A5F7A; margin-bottom:6px; }
.icu-kpi-value { font-family:'JetBrains Mono',monospace; font-size:28px; font-weight:700; color:#0D1B2A; line-height:1; }
.icu-kpi-sub { font-family:'JetBrains Mono',monospace; font-size:11px; color:#3A5F7A; margin-top:4px; }
.patient-card { background:linear-gradient(135deg,#DDF0F9 0%,#D0E8F5 100%); border:1px solid #90BDD8; border-radius:10px; padding:22px 26px; margin-bottom:20px; }
.icu-badge { display:inline-flex; align-items:center; padding:4px 14px; border-radius:4px; font-family:'JetBrains Mono',monospace; font-size:11px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; }
.badge-LOW      { background:#D4EDDA; color:#155724; border:1px solid #A3D4AE; }
.badge-MODERATE { background:#FFF3CD; color:#856404; border:1px solid #F0C040; }
.badge-HIGH     { background:#FFE0C2; color:#7A3200; border:1px solid #F0A060; }
.badge-SEVERE   { background:#F8D7DA; color:#721C24; border:1px solid #F0A0A4; }
.badge-CRITICAL { background:#E8D5F5; color:#4A1080; border:1px solid #C090E0; }
.icu-alert { border-left:3px solid; padding:10px 16px; margin:6px 0; border-radius:0 6px 6px 0; font-family:'Syne',sans-serif; font-size:13px; font-weight:500; line-height:1.5; }
.alert-critical { border-color:#6A1B9A; background:rgba(106,27,154,0.08); color:#4A1070; }
.alert-high     { border-color:#C62828; background:rgba(198,40,40,0.08); color:#7F0000; }
.alert-warning  { border-color:#C8920A; background:rgba(200,146,10,0.08);  color:#7A5000; }
.alert-info     { border-color:#1565C0; background:rgba(21,101,192,0.08);  color:#0D3C7A; }
.alert-ok       { border-color:#1B8839; background:rgba(27,136,57,0.08);   color:#0A4020; }
.icu-risk-bar { margin-bottom:16px; }
.icu-risk-bar-header { display:flex; justify-content:space-between; align-items:center; margin-bottom:6px; }
.icu-risk-bar-name { font-family:'Syne',sans-serif; font-size:14px; font-weight:600; color:#1E3448; }
.icu-risk-bar-track { background:#B5D8EC; border-radius:3px; height:7px; overflow:hidden; }
.icu-risk-bar-fill { height:100%; border-radius:3px; transition:width 0.5s ease; }

/* MODALITY CARDS */
.icu-modality {
    background:var(--bg-card); border:1px solid var(--border); border-radius:8px;
    padding:20px 16px; text-align:center; transition:border-color 0.2s, transform 0.2s, box-shadow 0.2s;
    cursor:pointer;
}
.icu-modality:hover { transform:translateY(-3px); border-color:var(--accent-blue); box-shadow:0 0 16px rgba(21,101,192,0.15); }

.shap-info-box { background:rgba(21,101,192,0.06); border:1px solid rgba(21,101,192,0.2); border-radius:6px; padding:12px 16px; font-family:'Syne',sans-serif; font-size:13px; color:#0D3C7A; margin-bottom:16px; line-height:1.6; }
.sandbox-card { background:var(--bg-card); border:1px solid var(--border); border-radius:8px; padding:18px 20px; margin-bottom:12px; transition:border-color 0.2s; }
.sandbox-card:hover { border-color:var(--border-mid); }
.sandbox-title { font-family:'Syne',sans-serif; font-size:14px; font-weight:700; color:#0D1B2A; margin-bottom:4px; }
.sandbox-desc { font-family:'JetBrains Mono',monospace; font-size:10px; color:#3A5F7A; letter-spacing:0.08em; margin-bottom:12px; }
.delta-card { border-radius:8px; padding:14px 16px; text-align:center; border:1px solid; }
.delta-label { font-family:'JetBrains Mono',monospace; font-size:10px; font-weight:700; letter-spacing:0.12em; text-transform:uppercase; color:#3A5F7A; margin-bottom:6px; }
.delta-before { font-family:'JetBrains Mono',monospace; font-size:13px; color:#3A5F7A; text-decoration:line-through; margin-bottom:2px; }
.delta-after { font-family:'JetBrains Mono',monospace; font-size:22px; font-weight:700; line-height:1; }
.delta-change { font-family:'JetBrains Mono',monospace; font-size:12px; font-weight:600; margin-top:4px; }
.timeline-item { display:flex; align-items:flex-start; gap:12px; padding:10px 0; border-bottom:1px solid #B5D8EC; }
.timeline-dot { width:10px; height:10px; border-radius:50%; margin-top:4px; flex-shrink:0; }
.timeline-time { font-family:'JetBrains Mono',monospace; font-size:11px; color:#3A5F7A; width:50px; flex-shrink:0; }
.timeline-text { font-family:'Syne',sans-serif; font-size:13px; color:#1E3448; line-height:1.4; }

@keyframes pulse-border {
    0%,100% { box-shadow: 0 0 0 0 rgba(106,27,154,0.3); }
    50%      { box-shadow: 0 0 0 8px rgba(106,27,154,0); }
}
</style>
""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════
# CONSTANTS
# ══════════════════════════════════════════════════
BASE      = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/Parquet')
MODEL_DIR = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/Models')
IMG_DIR   = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud')
FEEDBACK_PATH = BASE / 'icu_rlhf_feedback.parquet'
CXR_CACHE     = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/cxr_cache')
CXR_PATH_MAP  = BASE / 'icu_cxr_path_map.parquet'

TIER_COLORS = {'LOW':'#1B8839','MODERATE':'#C8920A','HIGH':'#D4500A','SEVERE':'#C62828','CRITICAL':'#6A1B9A'}
TIER_ORDER  = ['LOW','MODERATE','HIGH','SEVERE','CRITICAL']

PLOTLY_BASE = dict(
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='#D0E8F5',
    font=dict(family='JetBrains Mono', color='#1E3448', size=11),
    margin=dict(t=36, b=24, l=16, r=16),
)

# ══════════════════════════════════════════════════
# SESSION STATE — for tab navigation from modality cards
# ══════════════════════════════════════════════════
if 'active_modality_tab' not in st.session_state:
    st.session_state['active_modality_tab'] = 0

# ══════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════
@st.cache_data(show_spinner=False)
def get_hero_img():
    import glob
    candidates = (
        glob.glob(str(IMG_DIR / 'DashboardFunction' / '*.jpg')) +
        glob.glob(str(IMG_DIR / 'DashboardFunction' / '*.png')) +
        glob.glob(str(IMG_DIR / '*.jpg')) +
        glob.glob(str(IMG_DIR / '*.png'))
    )
    for p in candidates:
        try:
            from PIL import Image as PILImg, ImageFile
            ImageFile.LOAD_TRUNCATED_IMAGES = True
            import io
            img = PILImg.open(p); img.load()
            buf = io.BytesIO(); img.save(buf, format='JPEG', quality=85)
            return base64.b64encode(buf.getvalue()).decode()
        except Exception: continue
    return None

# ══════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════
@st.cache_data(show_spinner=False)
def load_data():
    static   = pd.read_parquet(BASE / 'icu_static_features.parquet')
    encoder  = pd.read_parquet(BASE / 'icu_encoder_features.parquet')
    risk     = pd.read_parquet(BASE / 'icu_risk_scores.parquet')
    preds    = pd.read_parquet(BASE / 'icu_fusion_predictions.parquet')
    text_emb = pd.read_parquet(BASE / 'icu_text_embeddings.parquet', columns=['stay_id','text_available'])
    cxr_emb  = pd.read_parquet(BASE / 'icu_cxr_embeddings.parquet',  columns=['stay_id','cxr_available'])

    fusion_meta, emb_meta = {}, {}
    if (MODEL_DIR / 'fusion_meta.json').exists():
        with open(MODEL_DIR / 'fusion_meta.json') as f: fusion_meta = json.load(f)
    if (MODEL_DIR / 'embedding_meta.json').exists():
        with open(MODEL_DIR / 'embedding_meta.json') as f: emb_meta = json.load(f)

    pred_cols = (
        ['stay_id','pred_sepsis_prob','pred_organ_failure_prob',
         'pred_mortality_prob','pred_criticality_tier','pred_criticality']
        + [c for c in preds.columns if c.startswith('pred_crit_prob_')]
        + ([c] if (c := next((x for x in ['split'] if x in preds.columns), None)) else [])
    )
    pred_cols = list(dict.fromkeys(pred_cols))

    risk_drop  = [c for c in risk.columns if c != 'stay_id' and c in static.columns]
    risk_clean = risk.drop(columns=risk_drop)
    master = static.merge(risk_clean, on='stay_id', how='left')
    master = master.merge(preds[[c for c in pred_cols if c in preds.columns]], on='stay_id', how='left')
    master = master.merge(text_emb, on='stay_id', how='left')
    master = master.merge(cxr_emb,  on='stay_id', how='left')

    shap_sep   = np.load(MODEL_DIR/'shap_sepsis.npy')        if (MODEL_DIR/'shap_sepsis.npy').exists()        else None
    shap_mort  = np.load(MODEL_DIR/'shap_mortality.npy')     if (MODEL_DIR/'shap_mortality.npy').exists()     else None
    shap_organ = np.load(MODEL_DIR/'shap_organ_failure.npy') if (MODEL_DIR/'shap_organ_failure.npy').exists() else None

    cxr_path_map = {}
    _cxr_pm = None
    if CXR_PATH_MAP.exists():
        pm = pd.read_parquet(CXR_PATH_MAP)
        _cxr_pm = pm.copy()
        if 'filename' not in pm.columns and 'path' in pm.columns:
            pm['filename'] = pm['path'].apply(lambda p: str(p).split('/')[-1])
        if 'filename' in pm.columns and 'stay_id' in pm.columns:
            for _, row in pm.iterrows():
                fpath = CXR_CACHE / row['filename']
                if fpath.exists():
                    cxr_path_map[int(row['stay_id'])] = fpath

    if _cxr_pm is not None and 'subject_id' in _cxr_pm.columns and 'stay_id' in _cxr_pm.columns:
        if 'filename' not in _cxr_pm.columns and 'path' in _cxr_pm.columns:
            _cxr_pm['filename'] = _cxr_pm['path'].apply(lambda p: str(p).split('/')[-1])
        if 'subject_id' in master.columns and 'filename' in _cxr_pm.columns:
            subj_to_icu = master[['stay_id','subject_id']].drop_duplicates()
            for _, mrow in subj_to_icu.iterrows():
                icu_sid = int(mrow['stay_id'])
                if icu_sid in cxr_path_map: continue
                subj = mrow['subject_id']
                pm_rows = _cxr_pm[_cxr_pm['subject_id'] == subj]
                for _, prow in pm_rows.iterrows():
                    fpath = CXR_CACHE / str(prow['filename'])
                    if fpath.exists():
                        cxr_path_map[icu_sid] = fpath; break

    return master, encoder, preds, fusion_meta, emb_meta, shap_sep, shap_mort, shap_organ, cxr_path_map

@st.cache_data(show_spinner=False)
def load_feedback():
    if FEEDBACK_PATH.exists(): return pd.read_parquet(FEEDBACK_PATH)
    return pd.DataFrame(columns=['timestamp','stay_id','doctor_id','prediction_agreed',
        'recommendation_useful','severity_rating','free_text','flag_for_review',
        'pred_sepsis_prob','pred_mortality_prob','pred_criticality_tier'])

def save_feedback(row: dict):
    df = load_feedback()
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_parquet(FEEDBACK_PATH, index=False)
    load_feedback.clear()

# ══════════════════════════════════════════════════
# RECOMMENDATIONS
# ══════════════════════════════════════════════════
RECS = {
    'sepsis_high': [
        ('🔬 Blood cultures ×2 before antibiotics', 'STAT', '#6A1B9A'),
        ('💊 Broad-spectrum IV antibiotics within 1 hour', 'STAT', '#6A1B9A'),
        ('💧 30 mL/kg IV crystalloid if MAP < 65 mmHg', 'URGENT', '#C62828'),
        ('📊 Serum lactate — repeat if > 2 mmol/L', 'URGENT', '#C62828'),
        ('🩺 Reassess fluid responsiveness post-bolus', 'IMPORTANT', '#C8920A'),
    ],
    'sepsis_mod': [
        ('📊 Trend lactate + procalcitonin', 'IMPORTANT', '#C8920A'),
        ('🩸 CBC, CMP, coagulation panel', 'IMPORTANT', '#C8920A'),
        ('🌡️ Temperature monitoring q4h', 'CONSIDER', '#1565C0'),
    ],
    'organ_high': [
        ('🫁 Lung-protective ventilation: TV 6 mL/kg IBW', 'STAT', '#6A1B9A'),
        ('🩺 Daily sedation holiday + spontaneous breathing trial', 'URGENT', '#C62828'),
        ('💉 Hold nephrotoxins — review and adjust GFR-based dosing', 'URGENT', '#C62828'),
        ('🧪 Serial troponin if cardiac dysfunction suspected', 'IMPORTANT', '#C8920A'),
        ('📋 Nephrology consult if AKI stage ≥ 2', 'IMPORTANT', '#C8920A'),
    ],
    'organ_mod': [
        ('📊 Urine output q1h — target > 0.5 mL/kg/hr', 'IMPORTANT', '#C8920A'),
        ('🩸 Daily BMP — trend creatinine trajectory', 'CONSIDER', '#1565C0'),
        ('💊 Comprehensive medication dosing review', 'CONSIDER', '#1565C0'),
    ],
    'mort_high': [
        ('👨‍👩‍👧 Goals of care discussion with patient and family', 'STAT', '#6A1B9A'),
        ('📋 Palliative care and ethics consult', 'URGENT', '#C62828'),
        ('🔄 Escalate monitoring — consider MICU transfer if HDU', 'URGENT', '#C62828'),
        ('📊 Daily SOFA scoring — track trajectory', 'IMPORTANT', '#C8920A'),
    ],
    'crit_critical': [
        ('🚨 Immediate bedside assessment', 'STAT', '#6A1B9A'),
        ('👨‍⚕️ Attending physician — notify NOW', 'STAT', '#6A1B9A'),
        ('📞 Rapid response team on standby', 'STAT', '#6A1B9A'),
        ('💉 Vasopressor readiness — confirm central access', 'URGENT', '#C62828'),
    ],
    'crit_severe': [
        ('👨‍⚕️ Physician review within 1 hour', 'URGENT', '#C62828'),
        ('📊 Continuous monitoring — vitals q15 min', 'URGENT', '#C62828'),
        ('🩸 Urgent labs: ABG, lactate, BMP, CBC', 'IMPORTANT', '#C8920A'),
    ],
}

def get_recs(patient):
    out  = []
    sep  = float(patient.get('pred_sepsis_prob', 0) or 0)
    org  = float(patient.get('pred_organ_failure_prob', 0) or 0)
    mort = float(patient.get('pred_mortality_prob', 0) or 0)
    tier = str(patient.get('pred_criticality_tier', 'LOW') or 'LOW')
    if sep  > 0.7:  out += [('Sepsis', *r) for r in RECS['sepsis_high']]
    elif sep > 0.4: out += [('Sepsis', *r) for r in RECS['sepsis_mod']]
    if org  > 0.7:  out += [('Organ Failure', *r) for r in RECS['organ_high']]
    elif org > 0.4: out += [('Organ Failure', *r) for r in RECS['organ_mod']]
    if mort > 0.5:  out += [('Mortality', *r) for r in RECS['mort_high']]
    if tier == 'CRITICAL': out += [('Criticality', *r) for r in RECS['crit_critical']]
    elif tier == 'SEVERE': out += [('Criticality', *r) for r in RECS['crit_severe']]
    priority_order = {'STAT':0,'URGENT':1,'IMPORTANT':2,'CONSIDER':3}
    out.sort(key=lambda x: priority_order.get(x[2], 9))
    return out

# ══════════════════════════════════════════════════
# PLOTLY HELPERS
# ══════════════════════════════════════════════════
def make_gauge(value, title, color, max_val=1.0, fmt='%'):
    dv = value * 100 if max_val == 1.0 else value
    dm = 100 if max_val == 1.0 else max_val
    fig = go.Figure(go.Indicator(
        mode='gauge+number', value=dv,
        number=dict(suffix=fmt, font=dict(size=28, color='#0D1B2A', family='JetBrains Mono')),
        title=dict(text=title, font=dict(size=13, color='#1E3448', family='JetBrains Mono')),
        gauge=dict(
            axis=dict(range=[0,dm], tickcolor='#90BDD8', tickfont=dict(color='#3A5F7A', size=10)),
            bar=dict(color=color, thickness=0.55), bgcolor='#C4E0F0',
            bordercolor='#90BDD8', borderwidth=1,
            steps=[dict(range=[0,dm*0.33],color='#D0E8F5'),dict(range=[dm*0.33,dm*0.66],color='#C4E0F0'),dict(range=[dm*0.66,dm],color='#B5D8EC')],
            threshold=dict(line=dict(color='#0D1B2A',width=2), thickness=0.75, value=dm*0.5),
        ),
    ))
    fig.update_layout(height=200, **PLOTLY_BASE)
    return fig

def make_shap_chart(shap_vals, feat_names, title, task_label, accent_color='#C62828'):
    sorted_idx   = np.argsort(np.abs(shap_vals))[-15:]
    sorted_shap  = shap_vals[sorted_idx]
    sorted_names = [feat_names[i][:35] for i in sorted_idx]
    colors = [TIER_COLORS['LOW'] if v < 0 else accent_color for v in sorted_shap]
    fig = go.Figure(go.Bar(
        x=sorted_shap, y=sorted_names, orientation='h',
        marker=dict(color=colors, line=dict(width=0)),
        text=[f'{v:+.4f}' for v in sorted_shap], textposition='outside',
        textfont=dict(family='JetBrains Mono', size=11, color='#0D1B2A'),
    ))
    fig.add_vline(x=0, line=dict(color='#90BDD8', width=1))
    fig.update_layout(
        title=dict(text=f'{title} — {task_label}', font=dict(size=14,color='#0D1B2A',family='Syne')),
        height=420,
        xaxis=dict(title='SHAP value',gridcolor='#90BDD8',zerolinecolor='#6CA8CC',tickfont=dict(color='#1E3448',size=11)),
        yaxis=dict(gridcolor='#90BDD8',tickfont=dict(color='#0D1B2A',size=12)),
        **PLOTLY_BASE,
    )
    return fig


# ══════════════════════════════════════════════════
# LOGIN PAGE
# ══════════════════════════════════════════════════
def render_login():
    img_b64 = get_hero_img()

    if img_b64:
        bg_css = f"""
        .stApp {{
            background-image: url('data:image/jpeg;base64,{img_b64}') !important;
            background-size: cover !important;
            background-position: center center !important;
            background-attachment: fixed !important;
        }}
        .stApp::before {{
            content: '';
            position: fixed; inset: 0;
            background: radial-gradient(ellipse at 50% 40%,
                rgba(235,244,251,0.82) 0%, rgba(221,240,249,0.96) 100%);
            z-index: 0; pointer-events: none;
        }}
        """
    else:
        bg_css = """
        .stApp {
            background: linear-gradient(135deg,#EBF4FB 0%,#DDF0F9 55%,#D0E8F5 100%) !important;
        }
        """

    st.markdown(f"""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&family=Syne:wght@700;800&display=swap');

    section[data-testid="stSidebar"] {{ display:none !important; }}
    header[data-testid="stHeader"]   {{ display:none !important; }}
    .block-container {{
        padding: 0 !important;
        max-width: 100% !important;
    }}

    {bg_css}

    .stTextInput > div > div > input {{
        background: rgba(255,255,255,0.85) !important;
        border: 1px solid rgba(21,101,192,0.30) !important;
        border-radius: 8px !important;
        color: #0D1B2A !important;
        font-family: 'JetBrains Mono', monospace !important;
        font-size: 14px !important;
        padding: 12px 16px !important;
        caret-color: #1565C0 !important;
    }}
    .stTextInput > div > div > input::placeholder {{
        color: #90BDD8 !important;
    }}
    .stTextInput > div > div > input:focus {{
        border-color: #1565C0 !important;
        box-shadow: 0 0 0 3px rgba(21,101,192,0.18) !important;
        background: rgba(255,255,255,0.95) !important;
    }}
    .stTextInput > div > div > div > button {{
        color: #1565C0 !important;
        background: transparent !important;
        border: none !important;
    }}

    div[data-testid="stButton"] button[kind="primary"] {{
        background: linear-gradient(135deg,#1565C0 0%,#1976D2 100%) !important;
        border: 1px solid rgba(21,101,192,0.45) !important;
        border-radius: 8px !important;
        color: #FFFFFF !important;
        font-family: 'Syne', sans-serif !important;
        font-size: 15px !important;
        font-weight: 700 !important;
        padding: 12px !important;
        letter-spacing: 0.04em !important;
        transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button[kind="primary"]:hover {{
        background: linear-gradient(135deg,#1976D2 0%,#1E88E5 100%) !important;
        box-shadow: 0 0 24px rgba(21,101,192,0.30) !important;
    }}

    .login-card-shell {{
        background: rgba(255,255,255,0.80);
        border: 1px solid rgba(21,101,192,0.22);
        border-radius: 16px;
        padding: 44px 48px 36px;
        width: 100%;
        backdrop-filter: blur(20px);
        box-shadow: 0 32px 80px rgba(21,101,192,0.12), 0 0 0 1px rgba(21,101,192,0.06);
        position: relative; z-index: 1;
    }}
    .login-logo-text {{
        font-family: 'Syne', sans-serif;
        font-size: 36px;
        font-weight: 800;
        color: #0D1B2A;
        letter-spacing: -0.02em;
        line-height: 1.1;
        text-align: center;
        margin-bottom: 8px;
    }}
    .login-sub-text {{
        font-family: 'JetBrains Mono', monospace;
        font-size: 10px;
        font-weight: 600;
        color: #1565C0;
        letter-spacing: 0.16em;
        text-transform: uppercase;
        text-align: center;
        margin-bottom: 36px;
    }}
    .login-divider {{
        border: none;
        border-top: 1px solid rgba(21,101,192,0.15);
        margin: 28px 0;
    }}
    .login-field-lbl {{
        font-family: 'JetBrains Mono', monospace;
        font-size: 10px;
        font-weight: 700;
        letter-spacing: 0.16em;
        text-transform: uppercase;
        color: #3A5F7A;
        margin-bottom: 6px;
    }}
    .login-hint {{
        font-family: 'JetBrains Mono', monospace;
        font-size: 9px;
        color: #90BDD8;
        text-align: center;
        margin-top: 20px;
        line-height: 1.7;
        letter-spacing: 0.04em;
    }}
    </style>
    """, unsafe_allow_html=True)

    st.markdown('<div style="height:6vh;"></div>', unsafe_allow_html=True)

    _, center, _ = st.columns([1, 1.4, 1])
    with center:
        st.markdown("""
        <div class="login-card-shell">
            <div class="login-logo-text">🏥 ICU AI Assistant</div>
            <div class="login-sub-text">
                Clinical Decision Support &nbsp;·&nbsp; MIMIC-IV &nbsp;·&nbsp; Research
            </div>
            <hr class="login-divider">
        </div>
        """, unsafe_allow_html=True)

        st.markdown('<div class="login-field-lbl">Doctor ID</div>', unsafe_allow_html=True)
        uid = st.text_input(
            label='Doctor ID',
            placeholder='e.g. DR001',
            key='login_uid',
            label_visibility='collapsed',
        )

        st.markdown('<div style="height:6px;"></div>', unsafe_allow_html=True)

        st.markdown('<div class="login-field-lbl">Password</div>', unsafe_allow_html=True)
        pwd = st.text_input(
            label='Password',
            placeholder='Enter password',
            type='password',
            key='login_pwd',
            label_visibility='collapsed',
        )

        st.markdown('<div style="height:14px;"></div>', unsafe_allow_html=True)

        if st.session_state.get('login_error'):
            st.markdown(f"""
            <div style="background:rgba(198,40,40,0.08);border:1px solid rgba(198,40,40,0.30);
                        border-radius:7px;padding:10px 14px;margin-bottom:12px;
                        font-family:'JetBrains Mono',monospace;font-size:12px;color:#C62828;">
                ⚠&nbsp; {st.session_state['login_error']}
            </div>""", unsafe_allow_html=True)

        if st.button('🔐  Login', type='primary', use_container_width=True, key='login_btn'):
            uid_clean = uid.strip().upper()
            if uid_clean in VALID_CREDENTIALS and VALID_CREDENTIALS[uid_clean] == pwd.strip():
                st.session_state['logged_in']   = True
                st.session_state['doctor_id']   = uid_clean
                st.session_state['login_error'] = ''
                st.rerun()
            else:
                st.session_state['login_error'] = 'Invalid Doctor ID or password. Try DR001 / icu2024'
                st.rerun()

        st.markdown("""
        <div class="login-hint">
            MIMIC-IV &nbsp;·&nbsp; Research use only &nbsp;·&nbsp; Not for clinical decisions<br>
            Demo: <strong style="color:#6CA8CC;">DR001</strong> /
            <strong style="color:#6CA8CC;">icu2024</strong>
        </div>""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════
# SIDEBAR
# ══════════════════════════════════════════════════
def render_sidebar(master):
    st.sidebar.markdown("""
    <div style="padding:20px 16px 16px;border-bottom:1px solid #90BDD8;margin-bottom:8px;
                background:linear-gradient(135deg,rgba(21,101,192,0.05) 0%,transparent 100%);">
        <div style="font-family:'Syne',sans-serif;font-size:18px;font-weight:800;color:#0D1B2A;letter-spacing:0.02em;">🏥 ICU AI Assistant</div>
        <div style="font-family:'JetBrains Mono',monospace;font-size:10px;font-weight:600;color:#1565C0;letter-spacing:0.15em;text-transform:uppercase;margin-top:5px;">Clinical Decision Support · v1.0</div>
    </div>""", unsafe_allow_html=True)

    st.sidebar.markdown('<div style="font-family:\'JetBrains Mono\',monospace;font-size:10px;font-weight:700;letter-spacing:0.2em;text-transform:uppercase;color:#3A5F7A;padding:12px 16px 6px;">Patient Filter</div>', unsafe_allow_html=True)

    split_col = next((c for c in ['split','Split','dataset_split'] if c in master.columns), None)
    if split_col:
        splits    = ['All'] + sorted(master[split_col].dropna().unique().tolist())
        split_sel = st.sidebar.selectbox('Dataset Split', splits, index=0)
        df = master if split_sel == 'All' else master[master[split_col] == split_sel]
    else:
        df = master.copy()

    if 'pred_criticality_tier' in df.columns:
        tier_sel = st.sidebar.multiselect('Criticality Tier', TIER_ORDER, default=TIER_ORDER)
        if tier_sel: df = df[df['pred_criticality_tier'].isin(tier_sel)]

    search = st.sidebar.text_input('Search Stay ID', placeholder='e.g. 30020552')
    if search.strip():
        try: df = df[df['stay_id'] == int(search.strip())]
        except: pass

    if len(df) == 0:
        st.sidebar.warning('No patients match filters.')
        return None, None, None

    sort_opts = {
        'Mortality ↓':('pred_mortality_prob',False),'Sepsis ↓':('pred_sepsis_prob',False),
        'Organ Failure ↓':('pred_organ_failure_prob',False),'Criticality ↓':('pred_criticality',False),'Stay ID ↑':('stay_id',True),
    }
    sort_by = st.sidebar.selectbox('Sort By', list(sort_opts.keys()))
    scol, sasc = sort_opts[sort_by]
    if scol in df.columns: df = df.sort_values(scol, ascending=sasc)

    st.sidebar.markdown('<div style="font-family:\'JetBrains Mono\',monospace;font-size:10px;font-weight:700;letter-spacing:0.2em;text-transform:uppercase;color:#3A5F7A;padding:12px 16px 6px;">Select Patient</div>', unsafe_allow_html=True)
    options = df['stay_id'].head(300).tolist()

    def fmt(sid):
        return str(sid)

    selected = st.sidebar.selectbox(f'{min(len(options),300)} patients shown', options, format_func=fmt)

    n_total  = len(df)
    n_crit   = len(df[df['pred_criticality_tier'].isin(['CRITICAL','SEVERE'])]) if 'pred_criticality_tier' in df.columns else 0
    avg_mort = float(df['pred_mortality_prob'].mean()) if 'pred_mortality_prob' in df.columns else 0

    st.sidebar.markdown(f"""
    <div style="padding:0 16px 16px;">
        <div style="font-family:'JetBrains Mono',monospace;font-size:10px;font-weight:700;letter-spacing:0.2em;text-transform:uppercase;color:#3A5F7A;padding:12px 0 8px;">Cohort Summary</div>
        <div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;">
            <div style="background:#D0E8F5;border:1px solid #90BDD8;border-left:3px solid #1565C0;border-radius:0 6px 6px 0;padding:12px 14px;">
                <div style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#3A5F7A;text-transform:uppercase;letter-spacing:0.1em;">Total</div>
                <div style="font-family:'JetBrains Mono',monospace;font-size:22px;font-weight:700;color:#0D1B2A;">{n_total:,}</div>
            </div>
            <div style="background:#D0E8F5;border:1px solid #90BDD8;border-left:3px solid #C62828;border-radius:0 6px 6px 0;padding:12px 14px;">
                <div style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#3A5F7A;text-transform:uppercase;letter-spacing:0.1em;">Critical</div>
                <div style="font-family:'JetBrains Mono',monospace;font-size:22px;font-weight:700;color:#C62828;">{n_crit:,}</div>
            </div>
        </div>
        <div style="background:#D0E8F5;border:1px solid #90BDD8;border-left:3px solid #6A1B9A;border-radius:0 6px 6px 0;padding:12px 14px;margin-top:8px;">
            <div style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#3A5F7A;text-transform:uppercase;letter-spacing:0.1em;">Avg Mortality Risk</div>
            <div style="font-family:'JetBrains Mono',monospace;font-size:22px;font-weight:700;color:#6A1B9A;">{avg_mort*100:.1f}%</div>
        </div>
    </div>""", unsafe_allow_html=True)

    st.sidebar.divider()
    doctor_id = st.sidebar.text_input('Doctor ID', placeholder='e.g. DR-001')
    st.sidebar.markdown('<div style="padding:6px 16px;font-family:JetBrains Mono,monospace;font-size:10px;color:#90BDD8;">MIMIC-IV · Research use only</div>', unsafe_allow_html=True)
    return selected, df, doctor_id

# ══════════════════════════════════════════════════
# HERO BANNER
# ══════════════════════════════════════════════════
def render_hero(master):
    n_total  = len(master)
    n_crit   = len(master[master['pred_criticality_tier'].isin(['CRITICAL','SEVERE'])]) if 'pred_criticality_tier' in master.columns else 0
    avg_mort = float(master['pred_mortality_prob'].mean()) if 'pred_mortality_prob' in master.columns else 0
    avg_sep  = float(master['pred_sepsis_prob'].mean()) if 'pred_sepsis_prob' in master.columns else 0
    img_b64  = get_hero_img()
    img_tag  = f'<img class="hero-img" src="data:image/jpeg;base64,{img_b64}" />' if img_b64 else '<div class="hero-fallback"></div>'
    st.markdown(f"""
    <div class="hero-banner">{img_tag}
        <div class="hero-overlay"><div>
            <div class="hero-title">🏥 ICU AI Assistant</div>
            <div class="hero-sub">Multimodal Clinical Decision Support &nbsp;·&nbsp; MIMIC-IV &nbsp;·&nbsp; Research Use Only</div>
            <div class="hero-stats">
                <div><div class="hero-stat-val" style="color:#0D1B2A;">{n_total:,}</div><div class="hero-stat-label">ICU Stays</div></div>
                <div class="hero-divider"></div>
                <div><div class="hero-stat-val" style="color:#C62828;">{n_crit:,}</div><div class="hero-stat-label">Critical / Severe</div></div>
                <div class="hero-divider"></div>
                <div><div class="hero-stat-val" style="color:#6A1B9A;">{avg_mort*100:.1f}%</div><div class="hero-stat-label">Avg Mortality Risk</div></div>
                <div class="hero-divider"></div>
                <div><div class="hero-stat-val" style="color:#D4500A;">{avg_sep*100:.1f}%</div><div class="hero-stat-label">Avg Sepsis Risk</div></div>
            </div>
        </div></div>
    </div>""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════
# TAB 1 — PATIENT OVERVIEW
# ══════════════════════════════════════════════════
def tab_patient(patient, encoder):
    sid  = int(patient['stay_id'])
    tier = str(patient.get('pred_criticality_tier','N/A') or 'N/A')
    tc   = TIER_COLORS.get(tier,'#3A5F7A')
    _age = patient.get('age_icu', patient.get('anchor_age', patient.get('age', None)))
    try:    age = int(float(_age)) if _age is not None else '—'
    except: age = '—'
    gender  = patient.get('gender','—')
    gender  = 'Male' if gender=='M' else 'Female' if gender=='F' else str(gender)
    adm     = 'Elective' if patient.get('is_elective', False) else 'Emergency'
    intime  = str(patient.get('intime','—'))[:16]
    los_str = '—'
    try:
        _it = pd.to_datetime(patient.get('intime')); _ot = pd.to_datetime(patient.get('outtime'))
        if pd.notna(_it) and pd.notna(_ot): los_str = f'{(_ot-_it).total_seconds()/86400:.1f} days'
    except: pass

    sep  = float(patient.get('pred_sepsis_prob',0) or 0)
    org  = float(patient.get('pred_organ_failure_prob',0) or 0)
    mort = float(patient.get('pred_mortality_prob',0) or 0)

    st.markdown(f"""
    <div class="patient-card">
        <div style="display:flex;justify-content:space-between;align-items:flex-start;">
            <div>
                <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;letter-spacing:0.15em;text-transform:uppercase;margin-bottom:8px;">ICU Stay ID</div>
                <div style="font-family:'JetBrains Mono',monospace;font-size:36px;font-weight:700;color:#0D1B2A;letter-spacing:-0.02em;">{sid}</div>
                <div style="font-family:'Syne',sans-serif;font-size:15px;font-weight:500;color:#1E3448;margin-top:8px;">
                    {gender} &nbsp;·&nbsp; Age {age} &nbsp;·&nbsp; {adm} &nbsp;·&nbsp; Admitted {intime} &nbsp;·&nbsp; LOS {los_str}
                </div>
            </div>
            <div style="text-align:right;">
                <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;text-transform:uppercase;letter-spacing:0.15em;margin-bottom:10px;">Risk Tier</div>
                <span class="icu-badge badge-{tier}">{tier}</span>
                <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{tc};margin-left:10px;box-shadow:0 0 10px {tc};vertical-align:middle;"></span>
            </div>
        </div>
    </div>""", unsafe_allow_html=True)

    def rtl(v):
        if v > 0.75: return 'CRITICAL'
        if v > 0.50: return 'SEVERE'
        if v > 0.30: return 'HIGH'
        if v > 0.15: return 'MODERATE'
        return 'LOW'

    def _trend_arrow(prob, low_thresh=0.15, high_thresh=0.5):
        if prob is None: return '', '#3A5F7A'
        if prob > high_thresh: return '▲ WORSENING', '#C62828'
        if prob > low_thresh:  return '◆ ELEVATED',  '#C8920A'
        return '▼ IMPROVING', '#1B8839'

    c1,c2,c3,c4 = st.columns(4)
    for col, label, prob, ac in [(c1,'Sepsis Risk',sep,'#1565C0'),(c2,'Organ Failure',org,'#0097A7'),(c3,'Mortality Risk',mort,'#6A1B9A'),(c4,'Risk Tier',None,tc)]:
        with col:
            if prob is not None:
                rl = rtl(prob); rcolor = TIER_COLORS.get(rl,'#3A5F7A'); val = rl
                trend_txt, trend_clr = _trend_arrow(prob)
                st.markdown(f'''<div class="icu-kpi" style="--accent-color:{ac}">
                    <div class="icu-kpi-label">{label}</div>
                    <div class="icu-kpi-value" style="color:{rcolor};font-size:22px;">{val}</div>
                    <div style="font-family:JetBrains Mono,monospace;font-size:10px;font-weight:700;
                                color:{trend_clr};margin-top:6px;letter-spacing:0.06em;">{trend_txt}</div>
                    <div style="font-family:JetBrains Mono,monospace;font-size:11px;color:#3A5F7A;margin-top:2px;">{prob*100:.1f}%</div>
                </div>''', unsafe_allow_html=True)
            else:
                st.markdown(f'<div class="icu-kpi" style="--accent-color:{ac}"><div class="icu-kpi-label">{label}</div><div class="icu-kpi-value" style="color:{tc};font-size:22px;">{tier}</div></div>', unsafe_allow_html=True)

    st.markdown('<div class="icu-section-label">Risk Trend vs Cohort</div>', unsafe_allow_html=True)

    def _cohort_trend(val, col_name, df_ref):
        if val is None: return None, None, None, None
        try:
            cohort_med = float(df_ref[col_name].median()) if col_name in df_ref.columns else 0.5
        except: cohort_med = 0.5
        delta = val - cohort_med
        pct_delta = delta * 100
        if delta > 0.20:  return '🔴 WORSENING',  '#C62828', 'rgba(198,40,40,0.08)',  pct_delta
        if delta > 0.08:  return '🟠 ELEVATED',   '#D4500A', 'rgba(212,80,10,0.07)',  pct_delta
        if delta < -0.15: return '🟢 IMPROVING',  '#1B8839', 'rgba(27,136,57,0.08)',  pct_delta
        return              '🟡 STABLE',     '#C8920A', 'rgba(200,146,10,0.07)', pct_delta

    try:
        from streamlit import session_state as _ss
        _master_ref = _ss.get('_master_ref', encoder)
    except:
        _master_ref = encoder

    trend_items = [
        ('🦠 Sepsis',        sep,  'pred_sepsis_prob'),
        ('🫘 Organ Failure',  org,  'pred_organ_failure_prob'),
        ('💀 Mortality',      mort, 'pred_mortality_prob'),
    ]
    t1, t2, t3 = st.columns(3)
    for tcol, (label, val, col_key) in zip([t1,t2,t3], trend_items):
        trend_label, trend_color, trend_bg, pct_delta = _cohort_trend(val, col_key, _master_ref)
        if trend_label is None: continue
        sign = '+' if pct_delta > 0 else ''
        with tcol:
            st.markdown(f'''
            <div style="background:{trend_bg};border:1px solid {trend_color};border-radius:8px;
                        padding:14px 16px;position:relative;overflow:hidden;">
                <div style="font-family:JetBrains Mono,monospace;font-size:10px;font-weight:700;
                            letter-spacing:0.15em;text-transform:uppercase;color:#3A5F7A;margin-bottom:6px;">{label}</div>
                <div style="font-family:Syne,sans-serif;font-size:20px;font-weight:800;
                            color:{trend_color};letter-spacing:0.03em;">{trend_label}</div>
                <div style="font-family:JetBrains Mono,monospace;font-size:12px;color:{trend_color};
                            margin-top:6px;">{sign}{pct_delta:.1f}pp vs cohort median</div>
            </div>''', unsafe_allow_html=True)

    st.markdown('<div style="height:20px;"></div>', unsafe_allow_html=True)
    enc_row = encoder[encoder['stay_id'] == sid]
    cv, cl  = st.columns(2)

# ══════════════════════════════════════════════════
# TAB 2 — CLINICAL SCORES
# ══════════════════════════════════════════════════
def tab_scores(patient, encoder):
    sid = int(patient['stay_id'])
    st.markdown('<div class="icu-section-label">Rule-Based Clinical Scores</div>', unsafe_allow_html=True)
    apache2   = patient.get('apache2_score', patient.get('apsiii', None))
    sofa      = patient.get('sofa_score', patient.get('sofa', None))
    qsofa     = patient.get('qsofa_score', patient.get('qsofa', None))
    mort_rule = patient.get('apache2_pred_mortality', patient.get('pred_mortality_rule', None))

    def sc(val, mx):
        if val is None: return '#1B8839'
        pct = float(val)/mx
        if pct > 0.75: return '#C62828'
        if pct > 0.55: return '#D4500A'
        if pct > 0.35: return '#C8920A'
        if pct > 0.15: return '#1B8839'
        return '#2E7D32'

    g1,g2,g3,g4 = st.columns(4)
    for col, val, title, mx, sfx in [(g1,apache2,'APACHE II',71,''),(g2,sofa,'SOFA Score',24,''),(g3,qsofa,'qSOFA',3,''),(g4,mort_rule,'Pred Mortality',1.0,'%')]:
        color = sc(val, mx)
        with col:
            if val is not None: st.plotly_chart(make_gauge(float(val),title,color,mx,sfx), use_container_width=True)
            else: st.markdown(f'<div class="icu-kpi" style="--accent-color:{color};height:200px;display:flex;flex-direction:column;justify-content:center;"><div class="icu-kpi-label">{title}</div><div style="color:#3A5F7A;font-family:JetBrains Mono,monospace;font-size:14px;margin-top:10px;">Not available</div></div>', unsafe_allow_html=True)

    st.markdown('<div class="icu-section-label">Score Interpretation</div>', unsafe_allow_html=True)
    ic1,ic2 = st.columns(2)
    with ic1:
        if sofa is not None:
            sv = float(sofa)
            me = ('< 10%' if sv<6 else '15–20%' if sv<10 else '40–50%' if sv<15 else '> 50%')
            sc2 = '#1B8839' if sv<6 else '#C8920A' if sv<10 else '#C62828'
            st.markdown(f'<div class="icu-kpi" style="--accent-color:{sc2}"><div class="icu-kpi-label">SOFA — Estimated Hospital Mortality</div><div class="icu-kpi-value" style="color:{sc2};font-size:24px;">{me}</div><div class="icu-kpi-sub">Score {sv:.0f} / 24</div></div>', unsafe_allow_html=True)
    with ic2:
        if qsofa is not None:
            qv = float(qsofa)
            qi = 'Sepsis risk ELEVATED — further assessment required' if qv>=2 else 'Low sepsis screen'
            qc = '#C62828' if qv>=2 else '#1B8839'
            st.markdown(f'<div class="icu-kpi" style="--accent-color:{qc}"><div class="icu-kpi-label">qSOFA — Sepsis Screen Result</div><div class="icu-kpi-value" style="color:{qc};font-size:16px;">{qi}</div><div class="icu-kpi-sub">Score {qv:.0f} / 3</div></div>', unsafe_allow_html=True)

    st.markdown('<div class="icu-section-label">Organ System Status</div>', unsafe_allow_html=True)
    enc_row = encoder[encoder['stay_id'] == sid]
    if len(enc_row) > 0:
        enc = enc_row.iloc[0]
        organs = {'Respiratory':(['pao2fio2_min','spo2_min'],'🫁'),'Coagulation':(['platelet_min'],'🩸'),'Hepatic':(['bilirubin_max'],'🟡'),'Cardiovascular':(['meanbp_min'],'❤️'),'Neurological':(['gcs_min'],'🧠'),'Renal':(['creatinine_max'],'🫘')}
        organ_vals = {}
        for organ,(keys,icon) in organs.items():
            for k in keys:
                if k in enc.index and pd.notna(enc[k]): organ_vals[organ]=(float(enc[k]),icon); break
        if organ_vals:
            cols = st.columns(len(organ_vals))
            for col,(organ,(val,icon)) in zip(cols,organ_vals.items()):
                with col: st.markdown(f'<div class="icu-kpi" style="--accent-color:#1565C0;text-align:center;"><div style="font-size:24px;margin-bottom:6px;">{icon}</div><div class="icu-kpi-label">{organ}</div><div class="icu-kpi-value" style="font-size:20px;">{val:.1f}</div></div>', unsafe_allow_html=True)


# ══════════════════════════════════════════════════
# TAB 3 — AI PREDICTIONS
# ══════════════════════════════════════════════════
def tab_ai(patient, master, shap_sep, shap_mort, shap_organ, fusion_meta):
    sep  = float(patient.get('pred_sepsis_prob',0) or 0)
    org  = float(patient.get('pred_organ_failure_prob',0) or 0)
    mort = float(patient.get('pred_mortality_prob',0) or 0)

    st.markdown('<div class="icu-section-label">AI Fusion Model — Risk Probabilities</div>', unsafe_allow_html=True)
    col_bars, col_dist = st.columns([3,2])
    with col_bars:
        for name, val, color in [('Sepsis Onset (24–48h)',sep,'#C62828'),('Organ Failure',org,'#D4500A'),('Mortality (24–48h)',mort,'#6A1B9A')]:
            level = 'CRITICAL' if val>0.75 else 'HIGH' if val>0.5 else 'MODERATE' if val>0.25 else 'LOW'
            st.markdown(f"""
            <div class="icu-risk-bar">
                <div class="icu-risk-bar-header"><span class="icu-risk-bar-name">{name}</span><span class="icu-badge badge-{level}" style="font-size:10px;padding:2px 8px;">{level}</span></div>
                <div class="icu-risk-bar-track"><div class="icu-risk-bar-fill" style="width:{val*100:.1f}%;background:{color};"></div></div>
            </div>""", unsafe_allow_html=True)

    with col_dist:
        crit_probs = {t: float(patient.get(f'pred_crit_prob_{t}',0) or 0) for t in TIER_ORDER}
        if any(v>0 for v in crit_probs.values()):
            fig = go.Figure(go.Bar(x=list(crit_probs.values()),y=list(crit_probs.keys()),orientation='h',marker=dict(color=[TIER_COLORS.get(t,'#3A5F7A') for t in TIER_ORDER],line=dict(width=0)),text=[f'{v*100:.1f}%' for v in crit_probs.values()],textposition='outside',textfont=dict(family='JetBrains Mono',size=12,color='#0D1B2A')))
            fig.update_layout(title=dict(text='Criticality Distribution',font=dict(size=14,color='#0D1B2A',family='Syne')),height=240,xaxis=dict(range=[0,1.3],showgrid=False,showticklabels=False),yaxis=dict(gridcolor='#90BDD8',tickfont=dict(size=13,color='#0D1B2A')),showlegend=False,**PLOTLY_BASE)
            st.plotly_chart(fig, use_container_width=True)

    st.markdown('<div class="icu-section-label">SHAP Explainability</div>', unsafe_allow_html=True)
    task_registry = {}
    if shap_sep   is not None: task_registry['Sepsis']        = (shap_sep,   '#C62828')
    if shap_organ is not None: task_registry['Organ Failure'] = (shap_organ, '#D4500A')
    if shap_mort  is not None: task_registry['Mortality']     = (shap_mort,  '#6A1B9A')

    avail_labels = ' &nbsp;·&nbsp; '.join(f'<strong>{t}</strong>' for t in task_registry) if task_registry else '—'
    st.markdown(f"""
    <div class="shap-info-box">
        ℹ️ SHAP explanations for {avail_labels} — GradientExplainer · 100-patient test sample. &nbsp;
        <span style="color:#C62828;">Red</span>=Sepsis↑ &nbsp;|&nbsp;
        <span style="color:#D4500A;">Orange</span>=OrganFailure↑ &nbsp;|&nbsp;
        <span style="color:#6A1B9A;">Purple</span>=Mortality↑ &nbsp;|&nbsp;
        <span style="color:#1B8839;">Green</span>=decreases risk.
    </div>""", unsafe_allow_html=True)

    ts_cols   = fusion_meta.get('ts_cols',[])
    text_cols = fusion_meta.get('text_cols',[])
    cxr_cols  = fusion_meta.get('cxr_feat_cols',[]) + fusion_meta.get('cxr_prob_cols',[])
    all_feats = ts_cols + text_cols + cxr_cols

    if not task_registry:
        st.warning('No SHAP values found. Run the SHAP notebook (Step 12) first.')
    else:
        shap_task = st.selectbox('Select prediction to explain', list(task_registry.keys()), key='shap_task_selector')
        sv, task_accent = task_registry[shap_task]
        ts_end = len(ts_cols); text_end = ts_end + len(text_cols)
        sc1, sc2 = st.columns(2)

        with sc1:
            ts_imp = float(np.abs(sv[:,:ts_end]).mean())
            text_imp = float(np.abs(sv[:,ts_end:text_end]).mean())
            cxr_imp  = float(np.abs(sv[:,text_end:]).mean())
            total = ts_imp+text_imp+cxr_imp+1e-10
            fig = go.Figure(go.Bar(x=['Time Series','Clinical Text','Chest X-Ray'],y=[ts_imp,text_imp,cxr_imp],marker=dict(color=['#B5D8EC','#C4E0F0','#D0E8F5'],line=dict(color=['#1565C0','#1B8839','#6A1B9A'],width=2)),text=[f'{v/total*100:.1f}%' for v in [ts_imp,text_imp,cxr_imp]],textposition='outside',textfont=dict(family='JetBrains Mono',size=13,color='#0D1B2A')))
            fig.update_layout(title=dict(text=f'Modality Contribution — {shap_task}',font=dict(size=14,color='#0D1B2A',family='Syne')),height=300,xaxis=dict(gridcolor='#90BDD8',tickfont=dict(size=13,color='#0D1B2A')),yaxis=dict(gridcolor='#90BDD8',showticklabels=False),showlegend=False,**PLOTLY_BASE)
            st.plotly_chart(fig, use_container_width=True)

            if shap_task == 'Organ Failure' and ts_cols:
                st.markdown('<div class="icu-section-label" style="margin-top:16px;">Organ-System Feature Groups</div>', unsafe_allow_html=True)
                organ_kw = {'🫘 Renal':['creatinine','bun','urine'],'🫁 Respiratory':['spo2','pao2','fio2','resprate','peep'],'❤️ Cardiovascular':['heartrate','meanbp','sbp','dbp'],'🧠 Neurological':['gcs'],'🩸 Coagulation':['platelet','inr','pt','ptt'],'🟡 Hepatic':['bilirubin','alt','ast','alp']}
                organ_imps = {}
                msv = sv[:,:ts_end].mean(axis=0)
                for ol, kws in organ_kw.items():
                    idxs = [i for i,c in enumerate(ts_cols) if any(kw in c.lower() for kw in kws)]
                    if idxs: organ_imps[ol] = float(np.abs(msv[idxs]).mean())
                if organ_imps:
                    sorted_org = sorted(organ_imps.items(), key=lambda x: x[1], reverse=True)
                    organ_bar_colors = ['#C62828','#D4500A','#C8920A','#1B8839','#1565C0','#6A1B9A']
                    fig_org = go.Figure(go.Bar(
                        y=[o[0] for o in sorted_org],
                        x=[o[1] for o in sorted_org],
                        orientation='h',
                        marker=dict(
                            color=organ_bar_colors[:len(sorted_org)],
                            line=dict(width=0),
                            opacity=0.85,
                        ),
                        text=[f'{o[1]:.4f}' for o in sorted_org],
                        textposition='outside',
                        textfont=dict(family='JetBrains Mono',size=11,color='#0D1B2A'),
                    ))
                    fig_org.update_layout(title=dict(text='Mean |SHAP| by Organ System',font=dict(size=13,color='#0D1B2A',family='Syne')),height=280,xaxis=dict(gridcolor='#90BDD8',showticklabels=False),yaxis=dict(gridcolor='#90BDD8',tickfont=dict(color='#0D1B2A',size=12)),showlegend=False,**PLOTLY_BASE)
                    st.plotly_chart(fig_org, use_container_width=True)

        with sc2:
            mean_shap = sv.mean(axis=0)
            if all_feats and len(all_feats)==len(mean_shap):
                st.plotly_chart(make_shap_chart(mean_shap,all_feats,'Top Feature Contributions',shap_task,accent_color=task_accent), use_container_width=True)
            else:
                st.info(f'Feature alignment mismatch (feats={len(all_feats)}, shap={len(mean_shap)}).')

        if len(task_registry)==3:
            st.markdown('<div class="icu-section-label">Cross-Task SHAP Comparison</div>', unsafe_allow_html=True)
            if all_feats and len(all_feats)==shap_sep.shape[1]:
                mean_imp = {'Sepsis':np.abs(shap_sep).mean(axis=0),'Organ Failure':np.abs(shap_organ).mean(axis=0),'Mortality':np.abs(shap_mort).mean(axis=0)}
                max_imp  = np.stack(list(mean_imp.values())).max(axis=0)
                top_idx  = np.argsort(max_imp)[-12:][::-1]
                top_names= [all_feats[i][:30] for i in top_idx]
                fig_cmp  = go.Figure()
                for tn, tc in [('Sepsis','#C62828'),('Organ Failure','#D4500A'),('Mortality','#6A1B9A')]:
                    fig_cmp.add_trace(go.Bar(name=tn,y=top_names,x=[mean_imp[tn][i] for i in top_idx],orientation='h',marker=dict(color=tc,opacity=0.85,line=dict(width=0))))
                fig_cmp.update_layout(barmode='group',title=dict(text='Top 12 Features — All Tasks',font=dict(size=14,color='#0D1B2A',family='Syne')),height=440,xaxis=dict(title='Mean |SHAP|',gridcolor='#90BDD8',tickfont=dict(color='#1E3448',size=10)),yaxis=dict(gridcolor='#90BDD8',tickfont=dict(color='#0D1B2A',size=11)),legend=dict(font=dict(color='#1E3448',size=12),bgcolor='rgba(0,0,0,0)',orientation='h',yanchor='bottom',y=1.02,xanchor='right',x=1),**PLOTLY_BASE)
                st.plotly_chart(fig_cmp, use_container_width=True)

    st.markdown('<div class="icu-section-label">Model Performance — Test Set</div>', unsafe_allow_html=True)
    test_m = fusion_meta.get('test_metrics',{})
    mm1,mm2,mm3,mm4 = st.columns(4)
    for col,label,val,color in [(mm1,'Sepsis AUROC',test_m.get('sepsis_auroc',0),'#C62828'),(mm2,'Mortality AUROC',test_m.get('mortality_auroc',0),'#6A1B9A'),(mm3,'Organ Failure AUROC',test_m.get('organ_failure_auroc',0),'#D4500A'),(mm4,'Criticality Acc',test_m.get('criticality_acc',0),'#1565C0')]:
        with col: st.markdown(f'<div class="icu-kpi" style="--accent-color:{color}"><div class="icu-kpi-label">{label}</div><div class="icu-kpi-value" style="color:{color};font-size:24px;">{val:.4f}</div><div class="icu-kpi-sub">Test set</div></div>', unsafe_allow_html=True)

# ══════════════════════════════════════════════════
# TAB 4 — GRAD-CAM CXR
# ══════════════════════════════════════════════════
@st.cache_resource
def load_cxr_model():
    try:
        import torchxrayvision as xrv, torch
        model = xrv.models.DenseNet(weights='densenet121-res224-mimic_nb')
        model.eval()
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        return model.to(device), device
    except ImportError:
        return None, 'torchxrayvision_missing'
    except Exception:
        try:
            import torchxrayvision as xrv, torch
            model = xrv.models.DenseNet(weights='densenet121-res224-all')
            model.eval()
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
            return model.to(device), device
        except Exception:
            return None, None


# ══════════════════════════════════════════════════
# GRAD-CAM — FIXED VERSION
# ══════════════════════════════════════════════════
def compute_gradcam(img_array, model, device, target_class_idx=0):
    import torch, cv2
    from PIL import Image as PILImage

    img_2d = img_array.squeeze()
    if img_2d.ndim == 3:
        img_2d = img_2d[0]

    # CLAHE equalization with higher clip limit for portable/AP films
    img_norm = ((img_2d - img_2d.min()) / (img_2d.max() - img_2d.min() + 1e-8) * 255).astype(np.uint8)
    clahe    = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    img_eq   = clahe.apply(img_norm).astype(np.float32)
    img_eq   = (img_eq / 255.0) * 2048 - 1024

    tensor = torch.tensor(img_eq[np.newaxis, np.newaxis, :, :], dtype=torch.float32).to(device)
    tensor = torch.nn.functional.interpolate(tensor, size=(224, 224), mode='bilinear', align_corners=False)
    tensor.requires_grad_(False)

    # Storage dicts — cleaner than mutable lists
    gradients   = {}
    activations = {}

    def forward_hook(module, input, output):
        activations['value'] = output.detach().clone()

    def backward_hook(module, grad_input, grad_output):
        gradients['value'] = grad_output[0].detach().clone()

    # ── Target the deepest available conv layer for richest spatial signal ──
    # Try denseblock4 last dense layer first (best spatial resolution),
    # fall back to denseblock4 block, then denseblock3.
    target_layer = None
    for candidate in [
        lambda m: m.features.denseblock4.denselayer16.layers.conv2,
        lambda m: m.features.denseblock4.denselayer15.layers.conv2,
        lambda m: m.features.denseblock4,
        lambda m: m.features.denseblock3,
    ]:
        try:
            target_layer = candidate(model)
            break
        except AttributeError:
            continue

    if target_layer is None:
        # Ultimate fallback
        target_layer = model.features

    fwd_handle = target_layer.register_forward_hook(forward_hook)
    bwd_handle = target_layer.register_full_backward_hook(backward_hook)

    try:
        model.zero_grad()
        tensor_grad = tensor.requires_grad_(True)
        output      = model(tensor_grad)
        probs       = torch.sigmoid(output).squeeze()

        # Scalar score for target class — drives backward pass
        score = output[0, target_class_idx]
        model.zero_grad()
        score.backward(retain_graph=False)

    finally:
        fwd_handle.remove()
        bwd_handle.remove()

    # ── Fallback: hooks captured nothing (e.g. leaf module issue) ──
    if 'value' not in gradients or 'value' not in activations:
        cam = np.ones((224, 224), dtype=np.float32) * 0.5
        return cam, probs.cpu().detach().numpy()

    grads = gradients['value'].squeeze().cpu().numpy()    # (C, H, W) or (H, W)
    acts  = activations['value'].squeeze().cpu().numpy()  # (C, H, W) or (H, W)

    # ── Grad-CAM weighting ──
    # Use ALL gradient signs (not just positive) for weighting.
    # ReLU is applied to the FINAL cam, not the gradient weights.
    if grads.ndim == 3:
        weights = np.mean(grads, axis=(1, 2))   # shape: (C,)
        cam = np.zeros(acts.shape[1:], dtype=np.float32)
        for w, a in zip(weights, acts):
            cam += w * a
    elif grads.ndim == 2:
        # Single-channel activation map
        cam = grads * acts
    else:
        cam = np.zeros((7, 7), dtype=np.float32)

    # ReLU on final CAM only
    cam = np.maximum(cam, 0)

    # ── Robust normalisation ──
    # If the CAM is nearly flat (model uncertain), fall back to
    # raw activation magnitude so we never return a blank heatmap.
    cam_min, cam_max = cam.min(), cam.max()
    if cam_max - cam_min < 1e-6:
        # Activation-magnitude fallback
        if acts.ndim == 3:
            cam = np.mean(np.abs(acts), axis=0)
        else:
            cam = np.abs(acts)
        cam_min, cam_max = cam.min(), cam.max()

    if cam_max - cam_min > 1e-6:
        cam = (cam - cam_min) / (cam_max - cam_min)
    else:
        # Model truly has no spatial preference — return uniform mid heatmap
        cam = np.full_like(cam, 0.5)

    # Resize to 224×224
    cam_uint8 = (cam * 255).astype(np.uint8)
    cam = np.array(
        PILImage.fromarray(cam_uint8).resize((224, 224), PILImage.BILINEAR)
    ) / 255.0

    return cam, probs.cpu().detach().numpy()


def overlay_gradcam(orig_img_array, cam, alpha=0.4):
    import matplotlib.pyplot as plt, matplotlib.cm as cm, io
    from PIL import Image as PILImage
    orig = orig_img_array.squeeze()
    if orig.ndim==3: orig = orig[0]
    if orig.max()>0: orig = (orig-orig.min())/(orig.max()-orig.min())
    orig_224 = np.array(PILImage.fromarray((orig*255).astype(np.uint8)).resize((224,224)))/255.0
    heatmap  = cm.jet(cam)[:,:,:3]
    blended  = np.clip((1-alpha)*np.stack([orig_224]*3,axis=-1)+alpha*heatmap,0,1)
    fig, axes = plt.subplots(1,3,figsize=(15,5),facecolor='#EBF4FB')
    for ax,title,img in zip(axes,['Original X-Ray','Grad-CAM Heatmap','Overlay'],[np.stack([orig_224]*3,axis=-1),heatmap,blended]):
        ax.imshow(img); ax.set_title(title,color='#0D1B2A',fontsize=13,fontweight='bold'); ax.axis('off'); ax.set_facecolor('#EBF4FB')
    plt.tight_layout(pad=1.0)
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='#EBF4FB')
    plt.close(); buf.seek(0)
    return buf

def tab_gradcam(patient, cxr_path_map):
    import io
    from PIL import Image as PILImage
    sid       = int(patient['stay_id'])
    cxr_avail = bool(patient.get('cxr_available',0))

    st.markdown('<div class="icu-section-label">Grad-CAM — CXR Visual Explainability</div>', unsafe_allow_html=True)
    st.markdown("""
    <div style="font-family:'Syne',sans-serif;font-size:14px;color:#1E3448;margin-bottom:20px;line-height:1.6;">
        Grad-CAM highlights which regions of the chest X-ray DenseNet-121 focused on when predicting each pathology.
        <span style="color:#C62828;font-weight:600;">Red</span> = highest attention &nbsp;|&nbsp;
        <span style="color:#1565C0;font-weight:600;">Blue</span> = lowest attention.
    </div>""", unsafe_allow_html=True)

    if not cxr_avail:
        st.markdown('<div class="icu-alert alert-warning">🫁 No chest X-ray available for this stay. Select a patient with CXR available from the sidebar.</div>', unsafe_allow_html=True)
        return

    img_path = None
    if sid in cxr_path_map:
        c = Path(cxr_path_map[sid])
        if c.exists(): img_path = c
    if img_path is None and CXR_PATH_MAP.exists():
        try:
            pm = pd.read_parquet(CXR_PATH_MAP)
            if 'filename' not in pm.columns and 'path' in pm.columns:
                pm['filename'] = pm['path'].apply(lambda p: str(p).split('/')[-1])
            row = pm[pm['stay_id']==sid]
            if len(row)>0:
                c = CXR_CACHE / row.iloc[0]['filename']
                if c.exists(): img_path = c
        except: pass

    if img_path is None:
        no_cxr_html = (
            '<div style="background:rgba(21,101,192,0.06);border:1px solid rgba(21,101,192,0.2);'
            'border-radius:10px;padding:40px;text-align:center;margin-top:20px;">'
            '<div style="font-size:52px;margin-bottom:16px;">🫁</div>'
            '<div style="font-family:Syne,sans-serif;font-size:18px;font-weight:700;'
            'color:#0D1B2A;margin-bottom:10px;">No Chest X-Ray Available</div>'
            '<div style="font-family:JetBrains Mono,monospace;font-size:12px;color:#3A5F7A;line-height:2;">'
            f'This patient (Stay ID: <strong>{sid}</strong>) did not have a chest X-ray<br>'
            'recorded during this ICU admission in MIMIC-CXR.<br><br>'
            '<span style="color:#1565C0;font-weight:600;">'
            'Select a patient with CXR AVAILABLE from the sidebar to use Grad-CAM.</span>'
            '</div></div>'
        )
        st.markdown(no_cxr_html, unsafe_allow_html=True)
        return

    try:
        from PIL import ImageFile
        ImageFile.LOAD_TRUNCATED_IMAGES = True
        raw_bytes = img_path.read_bytes()
        img_pil   = PILImage.open(io.BytesIO(raw_bytes)).convert('L')
    except Exception as e:
        st.error(f'Failed to load image: {e}'); return

    img_w, img_h = img_pil.size
    aspect       = img_w / img_h
    is_portable  = (aspect < 0.85 or aspect > 1.25 or min(img_w,img_h) < 300)
    if is_portable:
        st.markdown('<div class="icu-alert alert-warning" style="margin-bottom:16px;">⚠️ <strong>Portable / Semi-erect X-ray detected</strong> — CLAHE equalization and denseblock hooks applied. Interpret with caution.</div>', unsafe_allow_html=True)

    with st.spinner('Loading DenseNet-121 (MIMIC weights)...'):
        model, device = load_cxr_model()

    PATHOLOGY_FALLBACK = ['Atelectasis','Cardiomegaly','Consolidation','Edema','Effusion',
                          'Emphysema','Fibrosis','Hernia','Infiltration','Mass','Nodule',
                          'Pleural_Thickening','Pneumonia','Pneumothorax']
    pathology_names = PATHOLOGY_FALLBACK
    if model is not None:
        try:
            _p = [str(x).strip() for x in model.pathologies if x is not None and str(x).strip() != '']
            if len(_p) >= 5: pathology_names = _p
        except: pass

    col_sel, col_info = st.columns([2,3])

    with col_sel:
        if model is None:
            st.warning('Model not loaded — cannot list pathologies.')
            selected_path = 'Atelectasis'
        else:
            selected_path = st.selectbox('Select pathology to explain', pathology_names, key='gradcam_pathology')
        target_idx = pathology_names.index(selected_path) if selected_path in pathology_names else 0

        prob_cols = [c for c in patient.index if c.startswith('cxr_prob_')]
        if prob_cols:
            st.markdown('<div class="icu-section-label">Precomputed Pathology Probs</div>', unsafe_allow_html=True)
            probs = {c.replace('cxr_prob_','').replace('_',' '): float(patient[c] or 0) for c in prob_cols}
            for pname, pval in sorted(probs.items(), key=lambda x: x[1], reverse=True)[:8]:
                color = '#C62828' if pval>0.3 else '#C8920A' if pval>0.15 else '#1B8839'
                st.markdown(f"""
                <div style="margin:5px 0;">
                    <div style="display:flex;justify-content:space-between;margin-bottom:3px;">
                        <span style="font-family:'Syne',sans-serif;font-size:13px;color:#1E3448;">{pname}</span>
                        <span style="font-family:'JetBrains Mono',monospace;font-size:13px;color:{color};font-weight:700;">{pval*100:.1f}%</span>
                    </div>
                    <div style="background:#B5D8EC;border-radius:3px;height:5px;">
                        <div style="background:{color};border-radius:3px;height:5px;width:{pval*100:.1f}%;"></div>
                    </div>
                </div>""", unsafe_allow_html=True)

    with col_info:
        model_label = 'mimic_nb + CLAHE' if is_portable else 'mimic_nb (ICU-optimised)'
        st.markdown(f"""
        <div class="icu-kpi" style="--accent-color:#0097A7;margin-bottom:12px;">
            <div class="icu-kpi-label">Image File</div>
            <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#1E3448;margin-top:4px;word-break:break-all;">{img_path.name}</div>
            <div style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#3A5F7A;margin-top:4px;">{len(raw_bytes)/1024:.0f} KB &nbsp;&middot;&nbsp; {img_w}&times;{img_h}px &nbsp;&middot;&nbsp; {model_label}</div>
        </div>""", unsafe_allow_html=True)

        st.markdown(f"""
        <div class="icu-kpi" style="--accent-color:#6A1B9A;">
            <div class="icu-kpi-label">Explaining Pathology</div>
            <div style="font-family:'JetBrains Mono',monospace;font-size:26px;font-weight:700;color:#6A1B9A;margin-top:6px;">{selected_path}</div>
            <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;margin-top:6px;">Grad-CAM &nbsp;&middot;&nbsp; denseblock4 &nbsp;&middot;&nbsp; percentile-clipped</div>
        </div>""", unsafe_allow_html=True)

    if st.button('🔍 Generate Grad-CAM', type='primary'):
        if model is None:
            if device == 'torchxrayvision_missing':
                st.markdown('<div class="icu-alert alert-warning">⚠️ torchxrayvision not installed.<br><code>pip install git+https://github.com/mlmed/torchxrayvision.git</code></div>', unsafe_allow_html=True)
            else:
                st.error('DenseNet-121 failed to load.')
        else:
            with st.spinner(f'Computing Grad-CAM for {selected_path}...'):
                try:
                    import torchxrayvision as xrv
                    img_arr    = np.array(img_pil).astype(np.float32)
                    img_arr    = xrv.datasets.normalize(img_arr, maxval=255, reshape=True)
                    cam, all_probs = compute_gradcam(img_arr, model, device, target_idx)
                    overlay_buf    = overlay_gradcam(img_arr, cam)

                    st.markdown('<div class="icu-section-label">Grad-CAM Result</div>', unsafe_allow_html=True)
                    st.image(overlay_buf, use_container_width=True)

                    selected_prob = float(all_probs[target_idx]) if target_idx < len(all_probs) else 0.0
                    prob_color    = '#C62828' if selected_prob>0.5 else '#C8920A' if selected_prob>0.25 else '#1B8839'
                    st.markdown(f"""
                    <div style="background:rgba(21,101,192,0.06);border:1px solid rgba(21,101,192,0.2);
                                border-radius:8px;padding:16px 20px;margin-top:12px;
                                display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:12px;">
                        <div>
                            <div style="font-family:'JetBrains Mono',monospace;font-size:10px;font-weight:700;
                                        letter-spacing:0.15em;text-transform:uppercase;color:#3A5F7A;margin-bottom:6px;">
                                Selected Pathology Result
                            </div>
                            <div style="font-family:'Syne',sans-serif;font-size:18px;font-weight:700;color:#0D1B2A;">
                                {selected_path}
                            </div>
                            <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;margin-top:4px;">
                                Red regions = DenseNet-121 attention focus for this prediction
                            </div>
                        </div>
                        <div style="text-align:center;">
                            <div style="font-family:'JetBrains Mono',monospace;font-size:36px;font-weight:700;color:{prob_color};">
                                {selected_prob*100:.1f}%
                            </div>
                            <div style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#3A5F7A;
                                        text-transform:uppercase;letter-spacing:0.1em;margin-top:4px;">
                                Probability
                            </div>
                        </div>
                    </div>""", unsafe_allow_html=True)

                    st.markdown(f"""
                    <div style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#3A5F7A;
                                margin-top:8px;padding:8px 0;">
                        Hook: denseblock4 &nbsp;·&nbsp; CLAHE: {'applied' if is_portable else 'standard'} &nbsp;·&nbsp; Weights: {model_label}
                    </div>""", unsafe_allow_html=True)

                except Exception as e:
                    st.error(f'Grad-CAM failed: {e}')
    else:
        st.markdown('<div class="icu-section-label">Raw Chest X-Ray</div>', unsafe_allow_html=True)
        st.image(img_pil, use_container_width=True, caption=f'Stay {sid} · {img_path.name}')
        st.info('👆 Select a pathology and click "Generate Grad-CAM"')

# ══════════════════════════════════════════════════
# TAB 5 — MODALITY DEEP DIVE
# ══════════════════════════════════════════════════
def tab_modality(patient, encoder, fusion_meta, cxr_path_map):
    sid = int(patient['stay_id'])

    st.markdown("""
    <div style="background:linear-gradient(135deg,rgba(21,101,192,0.06) 0%,rgba(0,151,167,0.04) 100%);
                border:1px solid rgba(21,101,192,0.2);border-radius:10px;padding:14px 20px;margin-bottom:20px;">
        <div style="font-family:'Syne',sans-serif;font-size:15px;font-weight:700;color:#0D1B2A;">
            📂 Modality Deep Dive — Stay {sid}
        </div>
        <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;margin-top:4px;">
            Explore raw features extracted from each input modality for this patient.
        </div>
    </div>""".format(sid=sid), unsafe_allow_html=True)

    default_tab = st.session_state.get('active_modality_tab', 0)
    if default_tab > 0:
        tab_hints = ['Time Series', 'Clinical Notes', 'Chest X-Ray']
        st.info(f'💡 Navigate to the **{tab_hints[default_tab]}** sub-tab below.')

    mod_tab1, mod_tab2, mod_tab3 = st.tabs(['⏱  Time Series Features', '📄  Clinical Notes', '🫁  Chest X-Ray Features'])

    with mod_tab1:
        st.markdown('<div class="icu-section-label">Encoder Feature Summary — Time Series</div>', unsafe_allow_html=True)
        enc_row = encoder[encoder['stay_id'] == sid]

        if len(enc_row) == 0:
            st.warning('No encoder features found for this stay.')
        else:
            enc = enc_row.iloc[0]
            st.markdown("""
            <div style="font-family:'Syne',sans-serif;font-size:13px;color:#1E3448;margin-bottom:12px;line-height:1.6;">
                Time-series features are aggregated statistics (mean, min, max, std, last) computed over the ICU stay from vitals, labs and flowsheet data, then encoded via an LSTM.
            </div>""", unsafe_allow_html=True)

            vital_groups = {
                '❤️ Cardiovascular': {
                    'Heart Rate (mean)': ['heartrate_mean'], 'Heart Rate (min)': ['heartrate_min'], 'Heart Rate (max)': ['heartrate_max'],
                    'MAP (mean)': ['meanbp_mean'], 'MAP (min)': ['meanbp_min'], 'SBP (mean)': ['sysbp_mean'],
                    'DBP (mean)': ['diasbp_mean'],
                },
                '🫁 Respiratory': {
                    'SpO₂ (mean)': ['spo2_mean'], 'SpO₂ (min)': ['spo2_min'],
                    'Resp Rate (mean)': ['resprate_mean'], 'Resp Rate (max)': ['resprate_max'],
                    'PaO₂ (mean)': ['pao2_mean'], 'FiO₂ (mean)': ['fio2_mean'],
                    'PaO₂/FiO₂ (min)': ['pao2fio2_min'],
                },
                '🧠 Neurological': {
                    'GCS (mean)': ['gcs_mean'], 'GCS (min)': ['gcs_min'],
                },
                '🫘 Renal / Labs': {
                    'Creatinine (mean)': ['creatinine_mean'], 'Creatinine (max)': ['creatinine_max'],
                    'BUN (mean)': ['bun_mean'], 'Lactate (mean)': ['lactate_mean'],
                    'WBC (mean)': ['wbc_mean'], 'Platelets (mean)': ['platelet_mean'],
                    'Bilirubin (max)': ['bilirubin_max'],
                },
                '🌡️ Temperature': {
                    'Temp °C (mean)': ['tempc_mean'], 'Temp °C (min)': ['tempc_min'], 'Temp °C (max)': ['tempc_max'],
                },
            }

            for group_name, fields in vital_groups.items():
                found = {}
                for label, keys in fields.items():
                    for k in keys:
                        if k in enc.index and pd.notna(enc[k]):
                            found[label] = round(float(enc[k]), 3); break
                if not found: continue

                st.markdown(f"""
                <div style="font-family:'JetBrains Mono',monospace;font-size:10px;font-weight:700;
                            letter-spacing:0.15em;text-transform:uppercase;color:#3A5F7A;
                            margin:16px 0 8px 0;padding-bottom:4px;border-bottom:1px solid #90BDD8;">
                    {group_name}
                </div>""", unsafe_allow_html=True)

                n_cols = min(len(found), 4)
                cols   = st.columns(n_cols)
                for i, (label, val) in enumerate(found.items()):
                    with cols[i % n_cols]:
                        st.markdown(f"""
                        <div class="icu-kpi" style="--accent-color:#1565C0;margin-bottom:8px;">
                            <div class="icu-kpi-label">{label}</div>
                            <div class="icu-kpi-value" style="font-size:22px;">{val}</div>
                        </div>""", unsafe_allow_html=True)

            ts_cols_meta = fusion_meta.get('ts_cols', [])
            if ts_cols_meta:
                with st.expander('📋 All Time-Series Feature Values', expanded=False):
                    ts_data = {}
                    for c in ts_cols_meta:
                        if c in enc.index and pd.notna(enc[c]):
                            ts_data[c] = float(enc[c])
                    if ts_data:
                        df_ts = pd.DataFrame(list(ts_data.items()), columns=['Feature','Value'])
                        df_ts['Value'] = df_ts['Value'].round(4)
                        st.dataframe(df_ts, use_container_width=True, height=300)
                    else:
                        st.info('No matched time-series features found.')

    with mod_tab2:
        st.markdown('<div class="icu-section-label">Clinical Notes — Stay-Level Data</div>', unsafe_allow_html=True)
        text_avail = bool(patient.get('text_available', 0))
        if not text_avail:
            st.markdown('<div class="icu-alert alert-warning">📄 No clinical text available for this patient stay.</div>', unsafe_allow_html=True)
        else:
            st.markdown("""
            <div style="background:rgba(27,136,57,0.06);border:1px solid rgba(27,136,57,0.2);
                        border-radius:8px;padding:12px 16px;margin-bottom:20px;
                        display:flex;align-items:center;gap:12px;">
                <span style="font-size:20px;">📄</span>
                <div>
                    <div style="font-family:'Syne',sans-serif;font-size:14px;font-weight:700;color:#1B8839;">
                        Clinical notes present — encoded via ClinicalBERT (768-dim)
                    </div>
                    <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;margin-top:3px;">
                        Discharge summaries · Progress notes · Included in ICU Fusion Layer
                    </div>
                </div>
            </div>""", unsafe_allow_html=True)

            HOURLY_PATH = BASE / 'icu_hourly_features_24h.parquet'
            hourly_df   = None
            if HOURLY_PATH.exists():
                try:
                    _hdf = pd.read_parquet(HOURLY_PATH)
                    if 'stay_id' in _hdf.columns:
                        hourly_df = _hdf[_hdf['stay_id'] == sid].copy()
                        if len(hourly_df) == 0: hourly_df = None
                except: hourly_df = None

            if hourly_df is not None and len(hourly_df) > 0:
                st.markdown('<div class="icu-section-label">24-Hour Hourly Vitals Trend</div>', unsafe_allow_html=True)
                hour_col = next((c for c in ['hour','charttime','time','hour_num','hrs'] if c in hourly_df.columns), None)
                if hour_col: hourly_df = hourly_df.sort_values(hour_col)

                vital_map = {
                    'Heart Rate':    ['heartrate','heart_rate','hr'],
                    'MAP':           ['meanbp','map','mean_bp','mean_arterial_pressure'],
                    'SpO₂':          ['spo2','oxygen_saturation','o2sat'],
                    'Resp Rate':     ['resprate','respiratory_rate','rr'],
                    'Temperature':   ['tempc','temperature','temp'],
                    'GCS':           ['gcs','glasgow_coma'],
                }
                found_vitals = {}
                for vname, candidates in vital_map.items():
                    for c in candidates:
                        match = next((col for col in hourly_df.columns if c in col.lower()), None)
                        if match and hourly_df[match].notna().sum() > 0:
                            found_vitals[vname] = match; break

                if found_vitals:
                    x_vals = (hourly_df[hour_col].tolist() if hour_col else list(range(len(hourly_df))))
                    vital_colors = {
                        'Heart Rate': '#C62828', 'MAP': '#1565C0', 'SpO₂': '#1B8839',
                        'Resp Rate':  '#6A1B9A', 'Temperature': '#D4500A', 'GCS': '#0097A7',
                    }
                    cv_vitals    = {k:v for k,v in found_vitals.items() if k in ['Heart Rate','MAP']}
                    resp_vitals  = {k:v for k,v in found_vitals.items() if k in ['SpO₂','Resp Rate']}
                    other_vitals = {k:v for k,v in found_vitals.items() if k not in cv_vitals and k not in resp_vitals}

                    def _make_trend(vitals_dict, title):
                        fig = go.Figure()
                        for vname, col in vitals_dict.items():
                            y_vals = hourly_df[col].tolist()
                            fig.add_trace(go.Scatter(x=x_vals, y=y_vals, mode='lines', name=vname,
                                line=dict(color=vital_colors.get(vname,'#1565C0'), width=2),
                                hovertemplate=f'<b>{vname}</b>: %{{y:.1f}}<extra></extra>'))
                        fig.update_layout(title=dict(text=title, font=dict(size=13,color='#0D1B2A',family='Syne')),
                            height=220,
                            xaxis=dict(title='Hour' if hour_col else 'Time Point', gridcolor='#90BDD8', tickfont=dict(color='#1E3448',size=10)),
                            yaxis=dict(gridcolor='#90BDD8', tickfont=dict(color='#1E3448',size=10)),
                            legend=dict(font=dict(color='#1E3448',size=11), bgcolor='rgba(0,0,0,0)', orientation='h', y=-0.3),
                            **PLOTLY_BASE)
                        return fig

                    tc1, tc2 = st.columns(2)
                    if cv_vitals:
                        with tc1: st.plotly_chart(_make_trend(cv_vitals, '❤️ Cardiovascular — HR & MAP'), use_container_width=True)
                    if resp_vitals:
                        with tc2: st.plotly_chart(_make_trend(resp_vitals, '🫁 Respiratory — SpO₂ & RR'), use_container_width=True)
                    if other_vitals:
                        st.plotly_chart(_make_trend(other_vitals, '🌡️ Other Vitals'), use_container_width=True)

                    st.markdown('<div class="icu-section-label" style="margin-top:4px;">Hourly Vital Summary</div>', unsafe_allow_html=True)
                    sum_cols = st.columns(min(len(found_vitals), 6))
                    for col, (vname, vcol) in zip(sum_cols, found_vitals.items()):
                        series = hourly_df[vcol].dropna()
                        if len(series) > 0:
                            clr = vital_colors.get(vname, '#1565C0')
                            with col:
                                st.markdown(f"""
                                <div class="icu-kpi" style="--accent-color:{clr};">
                                    <div class="icu-kpi-label">{vname}</div>
                                    <div style="font-family:'JetBrains Mono',monospace;font-size:13px;color:{clr};margin-top:4px;">
                                        <span style="color:#3A5F7A;font-size:10px;">Min</span> {series.min():.1f}<br>
                                        <span style="color:#3A5F7A;font-size:10px;">Avg</span> {series.mean():.1f}<br>
                                        <span style="color:#3A5F7A;font-size:10px;">Max</span> {series.max():.1f}
                                    </div>
                                </div>""", unsafe_allow_html=True)

                    with st.expander('📋 Raw Hourly Data Table', expanded=False):
                        display_cols = ([hour_col] if hour_col else []) + list(found_vitals.values())
                        display_df   = hourly_df[display_cols].copy()
                        new_cols = []
                        if hour_col:
                            sample_val = display_df.iloc[0, 0] if len(display_df) > 0 else 0
                            try:
                                if float(sample_val) > 100000:
                                    new_cols.append('Hour (index)')
                                else:
                                    new_cols.append(hour_col)
                            except:
                                new_cols.append(hour_col)
                        display_df.columns = new_cols + list(found_vitals.keys())
                        st.dataframe(display_df.round(2), use_container_width=True, height=300)
                else:
                    st.info('Hourly parquet found but no recognisable vital columns detected.')
            else:
                st.markdown('<div class="icu-section-label">Clinical Risk Scores from Notes</div>', unsafe_allow_html=True)
                st.markdown("""
                <div class="icu-alert alert-info">
                    ℹ️ <code>icu_hourly_features_24h.parquet</code> not found or no data for this stay.
                    Showing note-derived risk fields from <code>icu_risk_scores.parquet</code>.
                </div>""", unsafe_allow_html=True)

                note_fields = {
                    'Sepsis Alert':        ('sepsis_alert',       lambda v: ('YES','#C62828') if v else ('NO','#1B8839')),
                    'Organ Dysfunction':   ('organ_dysfunction',  lambda v: ('YES','#C62828') if v else ('NO','#1B8839')),
                    'Organ Failure Count': ('organ_failure_count',lambda v: (str(int(float(v))),'#C62828' if float(v)>=2 else '#C8920A' if float(v)==1 else '#1B8839')),
                    'Criticality Tier':    ('criticality_tier',   lambda v: (str(v), TIER_COLORS.get(str(v),'#3A5F7A'))),
                }
                nf_cols = st.columns(4)
                for col, (label, (field, fmt)) in zip(nf_cols, note_fields.items()):
                    raw = patient.get(field, None)
                    if raw is not None:
                        try: disp, clr = fmt(raw)
                        except: disp, clr = str(raw), '#1565C0'
                        with col:
                            st.markdown(f'<div class="icu-kpi" style="--accent-color:{clr}"><div class="icu-kpi-label">{label}</div><div class="icu-kpi-value" style="color:{clr};font-size:22px;">{disp}</div></div>', unsafe_allow_html=True)

            st.markdown("""
            <div style="background:rgba(255,255,255,0.5);border:1px solid #90BDD8;border-radius:6px;
                        padding:12px 16px;margin-top:16px;">
                <div style="font-family:'JetBrains Mono',monospace;font-size:10px;font-weight:700;
                            letter-spacing:0.15em;text-transform:uppercase;color:#3A5F7A;margin-bottom:8px;">
                    ClinicalBERT Encoding — Technical Details
                </div>
                <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:8px;">
                    <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#1E3448;">
                        <span style="color:#3A5F7A;display:block;font-size:9px;text-transform:uppercase;letter-spacing:0.1em;">Model</span>ClinicalBERT
                    </div>
                    <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#1E3448;">
                        <span style="color:#3A5F7A;display:block;font-size:9px;text-transform:uppercase;letter-spacing:0.1em;">Dimension</span>768
                    </div>
                    <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#1E3448;">
                        <span style="color:#3A5F7A;display:block;font-size:9px;text-transform:uppercase;letter-spacing:0.1em;">Training</span>Bio + Clinical corpus
                    </div>
                    <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#1E3448;">
                        <span style="color:#3A5F7A;display:block;font-size:9px;text-transform:uppercase;letter-spacing:0.1em;">Used in</span>ICU Fusion Layer
                    </div>
                </div>
            </div>""", unsafe_allow_html=True)

    with mod_tab3:
        st.markdown('<div class="icu-section-label">DenseNet-121 CXR Feature Summary</div>', unsafe_allow_html=True)
        cxr_avail = bool(patient.get('cxr_available', 0))

        if not cxr_avail:
            st.markdown('<div class="icu-alert alert-warning">🫁 No chest X-ray available for this patient stay.</div>', unsafe_allow_html=True)
        else:
            st.markdown("""
            <div style="font-family:'Syne',sans-serif;font-size:13px;color:#1E3448;margin-bottom:16px;line-height:1.6;">
                Chest X-rays were encoded using <strong>DenseNet-121</strong> — 1024 spatial features plus 18 pathology probability scores.
            </div>""", unsafe_allow_html=True)

            prob_cols = [c for c in patient.index if c.startswith('cxr_prob_')]
            if prob_cols:
                st.markdown('<div class="icu-section-label" style="margin-top:20px;">Pathology Probability Scores</div>', unsafe_allow_html=True)
                probs = {c.replace('cxr_prob_','').replace('_',' '): float(patient[c] or 0) for c in prob_cols}
                sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)

                top_probs = sorted_probs[:6]
                rows = [top_probs[i:i+3] for i in range(0,len(top_probs),3)]
                for row in rows:
                    cols = st.columns(3)
                    for col,(pname,pval) in zip(cols,row):
                        clr = '#C62828' if pval>0.5 else '#C8920A' if pval>0.25 else '#1B8839'
                        with col: st.markdown(f'<div class="icu-kpi" style="--accent-color:{clr};margin-bottom:8px;"><div class="icu-kpi-label">{pname}</div><div class="icu-kpi-value" style="color:{clr};font-size:26px;">{pval*100:.1f}%</div></div>', unsafe_allow_html=True)

                st.markdown('<div class="icu-section-label" style="margin-top:16px;">All Pathology Scores — Ranked</div>', unsafe_allow_html=True)
                names_sorted = [p[0] for p in sorted_probs]
                vals_sorted  = [p[1] for p in sorted_probs]
                colors_sorted= ['#C62828' if v>0.5 else '#C8920A' if v>0.25 else '#1B8839' for v in vals_sorted]
                fig = go.Figure(go.Bar(y=names_sorted, x=vals_sorted, orientation='h',
                    marker=dict(color=colors_sorted, line=dict(width=0)),
                    text=[f'{v*100:.1f}%' for v in vals_sorted], textposition='outside',
                    textfont=dict(family='JetBrains Mono',size=11,color='#0D1B2A')))
                fig.add_vline(x=0.5, line=dict(color='#C62828',width=1,dash='dash'))
                fig.update_layout(
                    title=dict(text='Pathology Probabilities (DenseNet-121)',font=dict(size=14,color='#0D1B2A',family='Syne')),
                    height=max(300, len(names_sorted)*28),
                    xaxis=dict(range=[0,1.2],showgrid=False,showticklabels=False),
                    yaxis=dict(gridcolor='#90BDD8',tickfont=dict(color='#0D1B2A',size=12)),
                    showlegend=False, **PLOTLY_BASE)
                st.plotly_chart(fig, use_container_width=True)
                st.markdown("""
                <div style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#3A5F7A;margin-top:4px;">
                    Dashed red line = 50% threshold &nbsp;·&nbsp; Precomputed during CXR encoding, not live inference
                </div>""", unsafe_allow_html=True)
            else:
                st.info('No precomputed CXR pathology probabilities found in parquet for this stay.')

            if sid in cxr_path_map:
                img_p = Path(cxr_path_map[sid])
                if img_p.exists():
                    st.markdown('<div class="icu-section-label" style="margin-top:20px;">Raw X-Ray Thumbnail</div>', unsafe_allow_html=True)
                    import io
                    from PIL import Image as PILImage, ImageFile
                    ImageFile.LOAD_TRUNCATED_IMAGES = True
                    try:
                        raw = img_p.read_bytes()
                        img = PILImage.open(io.BytesIO(raw)).convert('L')
                        st.image(img, width=320, caption=f'{img_p.name} · {img.width}×{img.height}px')
                    except Exception as _ie:
                        st.warning(f'Could not display thumbnail: {_ie}')
            else:
                no_thumb_html = (
                    '<div style="background:rgba(21,101,192,0.06);border:1px solid rgba(21,101,192,0.2);'
                    'border-radius:8px;padding:24px;text-align:center;margin-top:16px;">'
                    '<div style="font-size:36px;margin-bottom:10px;">🫁</div>'
                    '<div style="font-family:Syne,sans-serif;font-size:14px;font-weight:700;color:#0D1B2A;margin-bottom:6px;">'
                    'No X-Ray Image Available</div>'
                    f'<div style="font-family:JetBrains Mono,monospace;font-size:11px;color:#3A5F7A;">'
                    f'Stay {sid} has no chest X-ray in MIMIC-CXR for this admission.<br>'
                    'CXR embedding was computed from a related admission.</div>'
                    '</div>'
                )
                st.markdown(no_thumb_html, unsafe_allow_html=True)


# ══════════════════════════════════════════════════
# TAB 6 — PATIENT HISTORY
# ══════════════════════════════════════════════════
def tab_history(patient, master):
    sid        = int(patient['stay_id'])
    subject_id = patient.get('subject_id', None)

    st.markdown('''
    <div style="background:linear-gradient(135deg,rgba(21,101,192,0.06) 0%,rgba(0,151,167,0.04) 100%);
                border:1px solid rgba(21,101,192,0.2);border-radius:10px;padding:14px 20px;margin-bottom:20px;">
        <div style="font-family:'Syne',sans-serif;font-size:15px;font-weight:700;color:#0D1B2A;">
            📋 Patient History
        </div>
        <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;margin-top:4px;">
            Past ICU admissions for this patient · Comorbidities · Diagnosis codes
        </div>
    </div>''', unsafe_allow_html=True)

    if subject_id is None:
        st.warning("No subject_id found for this stay — cannot link to patient history.")
        return

    all_stays = master[master['subject_id'] == subject_id].copy()
    all_stays = all_stays.sort_values('intime') if 'intime' in all_stays.columns else all_stays

    st.markdown('<div class="icu-section-label">ICU Admission History</div>', unsafe_allow_html=True)

    n_stays = len(all_stays)
    admit_col, current_col, first_col = st.columns(3)
    with admit_col:
        st.markdown(f'''<div class="icu-kpi" style="--accent-color:#1565C0;">
            <div class="icu-kpi-label">Total ICU Admissions</div>
            <div class="icu-kpi-value" style="font-size:32px;">{n_stays}</div>
        </div>''', unsafe_allow_html=True)
    with current_col:
        current_tier = str(patient.get('pred_criticality_tier','—') or '—')
        tc = TIER_COLORS.get(current_tier, '#3A5F7A')
        st.markdown(f'''<div class="icu-kpi" style="--accent-color:{tc};">
            <div class="icu-kpi-label">Current Stay Tier</div>
            <div class="icu-kpi-value" style="color:{tc};font-size:28px;">{current_tier}</div>
        </div>''', unsafe_allow_html=True)
    with first_col:
        first_admit = str(all_stays['intime'].min())[:10] if 'intime' in all_stays.columns else '—'
        st.markdown(f'''<div class="icu-kpi" style="--accent-color:#0097A7;">
            <div class="icu-kpi-label">First ICU Admission</div>
            <div class="icu-kpi-value" style="font-size:20px;color:#0097A7;">{first_admit}</div>
        </div>''', unsafe_allow_html=True)

    if n_stays > 0:
        st.markdown('<div class="icu-section-label" style="margin-top:20px;">Admission Timeline</div>', unsafe_allow_html=True)
        display_cols_wanted = ['stay_id','intime','outtime','pred_criticality_tier','pred_mortality_prob','pred_sepsis_prob','pred_organ_failure_prob']
        display_cols = [c for c in display_cols_wanted if c in all_stays.columns]
        tbl = all_stays[display_cols].copy()

        def _p2tier(v):
            if pd.isna(v): return '—'
            p = float(v)
            if p > 0.75: return 'CRITICAL'
            if p > 0.50: return 'SEVERE'
            if p > 0.30: return 'HIGH'
            if p > 0.15: return 'MODERATE'
            return 'LOW'
        for tc_col in ['pred_mortality_prob','pred_sepsis_prob','pred_organ_failure_prob']:
            if tc_col in tbl.columns:
                tbl[tc_col] = tbl[tc_col].apply(_p2tier)
        for dt_col in ['intime','outtime']:
            if dt_col in tbl.columns:
                tbl[dt_col] = tbl[dt_col].apply(lambda v: str(v)[:16] if pd.notna(v) else '—')

        rename_map = {
            'stay_id': 'Stay ID', 'intime': 'Admitted', 'outtime': 'Discharged',
            'pred_criticality_tier': 'Criticality', 'pred_mortality_prob': 'Mortality',
            'pred_sepsis_prob': 'Sepsis', 'pred_organ_failure_prob': 'Organ Failure'
        }
        tbl = tbl.rename(columns={k:v for k,v in rename_map.items() if k in tbl.columns})

        def _highlight_current(row):
            style = 'background-color: rgba(21,101,192,0.12); font-weight: bold;' if str(row.get('Stay ID','')) == str(sid) else ''
            return [style] * len(row)

        st.dataframe(tbl.style.apply(_highlight_current, axis=1), use_container_width=True, height=min(400, 50 + len(tbl)*38))
        st.markdown('<div style="font-family:JetBrains Mono,monospace;font-size:10px;color:#3A5F7A;margin-top:4px;">🔵 Highlighted row = current stay</div>', unsafe_allow_html=True)

        if 'intime' in all_stays.columns and 'outtime' in all_stays.columns:
            los_list = []
            for _, row in all_stays.iterrows():
                try:
                    it = pd.to_datetime(row['intime']); ot = pd.to_datetime(row['outtime'])
                    if pd.notna(it) and pd.notna(ot):
                        los_list.append({'Stay ID': str(int(row['stay_id'])), 'LOS (days)': round((ot-it).total_seconds()/86400, 1)})
                except: pass
            if los_list:
                st.markdown('<div class="icu-section-label" style="margin-top:20px;">Length of Stay per Admission</div>', unsafe_allow_html=True)
                los_df = pd.DataFrame(los_list)
                colors_los = ['#1565C0' if row['Stay ID'] != str(sid) else '#C62828' for _, row in los_df.iterrows()]
                fig_los = go.Figure(go.Bar(
                    x=los_df['Stay ID'], y=los_df['LOS (days)'],
                    marker=dict(color=colors_los, line=dict(width=0)),
                    text=los_df['LOS (days)'], textposition='outside',
                    textfont=dict(family='JetBrains Mono', size=12, color='#0D1B2A'),
                ))
                fig_los.update_layout(
                    title=dict(text='LOS per Admission (red = current stay)', font=dict(size=14,color='#0D1B2A',family='Syne')),
                    height=260,
                    xaxis=dict(title='Stay ID', gridcolor='#90BDD8', tickfont=dict(color='#0D1B2A', size=11)),
                    yaxis=dict(title='Days', gridcolor='#90BDD8', tickfont=dict(color='#0D1B2A', size=11)),
                    showlegend=False, **PLOTLY_BASE
                )
                st.plotly_chart(fig_los, use_container_width=True)

    risk_cols = [c for c in ['pred_mortality_prob','pred_sepsis_prob','pred_organ_failure_prob'] if c in all_stays.columns]
    if risk_cols and n_stays > 1 and 'intime' in all_stays.columns:
        st.markdown('<div class="icu-section-label" style="margin-top:8px;">Risk Score Trend Across Admissions</div>', unsafe_allow_html=True)
        fig_risk = go.Figure()
        risk_labels = {'pred_mortality_prob':'Mortality','pred_sepsis_prob':'Sepsis','pred_organ_failure_prob':'Organ Failure'}
        risk_colors = {'pred_mortality_prob':'#6A1B9A','pred_sepsis_prob':'#C62828','pred_organ_failure_prob':'#D4500A'}
        x_labels = [str(int(r['stay_id'])) for _,r in all_stays.iterrows()]
        def _to_tier_n(v):
            if pd.isna(v): return None
            p = float(v)
            if p > 0.75: return 4
            if p > 0.50: return 3
            if p > 0.30: return 2
            if p > 0.15: return 1
            return 0
        for rc in risk_cols:
            y_vals = [_to_tier_n(r[rc]) for _,r in all_stays.iterrows()]
            fig_risk.add_trace(go.Scatter(
                x=x_labels, y=y_vals, mode='lines+markers',
                name=risk_labels.get(rc, rc),
                line=dict(color=risk_colors.get(rc,'#1565C0'), width=2),
                marker=dict(size=8, symbol='circle'),
            ))
        fig_risk.update_layout(
            title=dict(text='Risk Tier Trend Across Admissions', font=dict(size=14,color='#0D1B2A',family='Syne')),
            height=300,
            xaxis=dict(title='Stay ID', gridcolor='#90BDD8', tickfont=dict(color='#0D1B2A',size=11)),
            yaxis=dict(title='Risk Tier', gridcolor='#90BDD8', tickfont=dict(color='#0D1B2A',size=11),
                       range=[-0.5,4.5], tickvals=[0,1,2,3,4],
                       ticktext=['LOW','MODERATE','HIGH','SEVERE','CRITICAL']),
            legend=dict(font=dict(color='#1E3448',size=12), bgcolor='rgba(0,0,0,0)', orientation='h', y=-0.3),
            **PLOTLY_BASE
        )
        st.plotly_chart(fig_risk, use_container_width=True)

    st.markdown('<div class="icu-section-label" style="margin-top:8px;">Comorbidities & Static Features</div>', unsafe_allow_html=True)

    comorbidity_fields = {
        '🫀 Congestive Heart Failure':  ['chf','congestive_heart_failure','heart_failure'],
        '🫁 COPD / Lung Disease':       ['copd','chronic_pulmonary','lung_disease'],
        '🩸 Diabetes':                  ['diabetes','diabetes_mellitus','dm','diabetes_uncomplicated','diabetes_complicated'],
        '🧠 Dementia':                  ['dementia'],
        '🦠 HIV / AIDS':                ['aids','hiv'],
        '🫘 Renal Disease':             ['renal','renal_failure','kidney_disease','ckd'],
        '🟡 Liver Disease':             ['liver','liver_disease','cirrhosis'],
        '🎗 Malignancy':               ['malignancy','cancer','tumor','metastatic'],
        '❤️ Hypertension':             ['hypertension','htn'],
        '💔 MI / Cardiac History':      ['mi','myocardial_infarction','cardiac'],
        '🩺 Cerebrovascular Disease':   ['cerebrovascular','stroke','cva'],
        '🦵 Peripheral Vascular':       ['pvd','peripheral_vascular'],
    }

    found_comorbidities = {}
    all_cols_lower = {c.lower(): c for c in patient.index}
    for display_name, keywords in comorbidity_fields.items():
        for kw in keywords:
            matched_col = next((orig for lwr,orig in all_cols_lower.items() if kw in lwr), None)
            if matched_col and pd.notna(patient.get(matched_col)):
                val = patient.get(matched_col)
                try:
                    val_f = float(val)
                    if val_f > 0:
                        found_comorbidities[display_name] = (matched_col, val_f)
                        break
                except:
                    if str(val).lower() in ('true','1','yes'):
                        found_comorbidities[display_name] = (matched_col, 1.0)
                        break

    if found_comorbidities:
        n_comorbid = len(found_comorbidities)
        st.markdown(f'''
        <div style="background:rgba(198,40,40,0.06);border:1px solid rgba(198,40,40,0.2);
                    border-radius:8px;padding:12px 18px;margin-bottom:16px;
                    display:flex;align-items:center;gap:16px;">
            <div style="font-family:'JetBrains Mono',monospace;font-size:32px;font-weight:700;color:#C62828;">{n_comorbid}</div>
            <div>
                <div style="font-family:'Syne',sans-serif;font-size:14px;font-weight:700;color:#0D1B2A;">Comorbidities Detected</div>
                <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;margin-top:3px;">
                    Higher comorbidity burden increases ICU risk
                </div>
            </div>
        </div>''', unsafe_allow_html=True)

        comorbid_cols = st.columns(min(4, len(found_comorbidities)))
        for i, (name, (col, val)) in enumerate(found_comorbidities.items()):
            with comorbid_cols[i % 4]:
                st.markdown(f'''<div class="icu-kpi" style="--accent-color:#C62828;margin-bottom:8px;">
                    <div class="icu-kpi-label">{name}</div>
                    <div style="font-family:'JetBrains Mono',monospace;font-size:18px;font-weight:700;
                                color:#C62828;margin-top:4px;">PRESENT</div>
                    <div style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#3A5F7A;margin-top:2px;">{col}</div>
                </div>''', unsafe_allow_html=True)
    else:
        st.info("No comorbidity fields found in the static features parquet. Check your column names — expected fields like 'chf', 'copd', 'diabetes', etc.")

    st.markdown('<div class="icu-section-label" style="margin-top:20px;">Patient Demographics</div>', unsafe_allow_html=True)
    demo_fields = {
        'Age':          ['age_icu','anchor_age','age'],
        'Gender':       ['gender'],
        'Admission Type':['is_elective','admission_type'],
        'Ethnicity':    ['ethnicity','race'],
        'Insurance':    ['insurance'],
        'Marital Status':['marital_status'],
    }
    demo_found = {}
    for label, keys in demo_fields.items():
        for k in keys:
            v = patient.get(k, None)
            if v is not None and pd.notna(v):
                if label == 'Gender': v = 'Male' if str(v)=='M' else 'Female' if str(v)=='F' else str(v)
                if label == 'Admission Type': v = 'Elective' if v else 'Emergency'
                demo_found[label] = str(v)[:30]
                break

    if demo_found:
        demo_cols = st.columns(min(4, len(demo_found)))
        demo_colors = ['#1565C0','#0097A7','#6A1B9A','#C8920A','#1B8839','#D4500A']
        for i,(label,val) in enumerate(demo_found.items()):
            clr = demo_colors[i % len(demo_colors)]
            with demo_cols[i % 4]:
                st.markdown(f'''<div class="icu-kpi" style="--accent-color:{clr};margin-bottom:8px;">
                    <div class="icu-kpi-label">{label}</div>
                    <div class="icu-kpi-value" style="font-size:18px;">{val}</div>
                </div>''', unsafe_allow_html=True)


# ══════════════════════════════════════════════════
# TAB 7 — RECOMMENDATIONS
# ══════════════════════════════════════════════════
def tab_recs(patient):
    recs = get_recs(patient)
    sep  = float(patient.get('pred_sepsis_prob',0) or 0)
    org  = float(patient.get('pred_organ_failure_prob',0) or 0)
    mort = float(patient.get('pred_mortality_prob',0) or 0)
    tier = str(patient.get('pred_criticality_tier','N/A') or 'N/A')
    tc   = TIER_COLORS.get(tier,'#3A5F7A')

    def risk_status(v):
        if v > 0.75: return 'CRITICAL',  '#6A1B9A','rgba(106,27,154,0.10)','#C090E0'
        if v > 0.50: return 'SEVERE',    '#C62828','rgba(198,40,40,0.10)', '#F0A0A4'
        if v > 0.25: return 'TRIGGERED', '#D4500A','rgba(212,80,10,0.08)', '#F0A060'
        if v > 0.10: return 'ELEVATED',  '#C8920A','rgba(200,146,10,0.07)','#F0C040'
        return             'NORMAL',    '#1B8839','rgba(27,136,57,0.07)', '#A3D4AE'

    st.markdown('<div class="icu-section-label">Risk Status</div>', unsafe_allow_html=True)
    cols = st.columns(3)
    for col,(icon,name,val) in zip(cols,[('🦠','Sepsis',sep),('🫘','Organ Failure',org),('💀','Mortality',mort)]):
        label,fg,bg,border = risk_status(val)
        triggered  = val > 0.25
        pulse_css  = 'animation:pulse-border 1.8s ease-in-out infinite;' if val>0.75 else ''
        traffic    = '🔴' if val>0.75 else '🟠' if val>0.50 else '🟡' if val>0.25 else '🟢'
        trig_color = fg if triggered else '#3A5F7A'
        trig_text  = '⚡ Recommendations triggered' if triggered else '✓ No recommendations triggered'
        with col:
            st.markdown(f"""
            <div style="background:{bg};border:1px solid {border};border-left:4px solid {fg};
                        border-radius:8px;padding:18px 20px;position:relative;{pulse_css}">
                <div style="display:flex;justify-content:space-between;align-items:flex-start;">
                    <div>
                        <div style="font-size:28px;line-height:1;margin-bottom:8px;">{icon}</div>
                        <div style="font-family:'JetBrains Mono',monospace;font-size:10px;font-weight:700;letter-spacing:0.15em;text-transform:uppercase;color:#3A5F7A;">{name}</div>
                        <div style="font-family:'JetBrains Mono',monospace;font-size:32px;font-weight:700;color:{fg};line-height:1;margin-top:6px;">{label}</div>
                    </div>
                    <div style="text-align:right;"><div style="font-size:24px;">{traffic}</div></div>
                </div>
                <div style="background:rgba(144,189,216,0.3);border-radius:3px;height:4px;margin-top:14px;overflow:hidden;">
                    <div style="background:{fg};height:4px;border-radius:3px;width:{min(val*100,100):.1f}%;"></div>
                </div>
                <div style="margin-top:10px;font-family:'JetBrains Mono',monospace;font-size:10px;color:{trig_color};letter-spacing:0.08em;">{trig_text}</div>
            </div>""", unsafe_allow_html=True)

    tier_desc = {'CRITICAL':'Immediate life-threatening — all STAT actions required NOW','SEVERE':'High severity — complete URGENT actions within 1 hour','HIGH':'Elevated risk — monitor closely this shift','MODERATE':'Moderate risk — standard enhanced monitoring','LOW':'Low risk — routine monitoring'}.get(tier,'')
    st.markdown(f"""
    <div style="display:flex;align-items:center;gap:14px;margin:16px 0 4px 0;padding:14px 18px;
                background:rgba(255,255,255,0.4);border:1px solid #90BDD8;border-left:4px solid {tc};border-radius:0 8px 8px 0;">
        <div style="font-family:'JetBrains Mono',monospace;font-size:11px;font-weight:700;letter-spacing:0.15em;text-transform:uppercase;color:#3A5F7A;white-space:nowrap;">Criticality Tier</div>
        <span class="icu-badge badge-{tier}">{tier}</span>
        <span style="width:10px;height:10px;border-radius:50%;background:{tc};box-shadow:0 0 10px {tc};display:inline-block;flex-shrink:0;"></span>
        <div style="font-family:'Syne',sans-serif;font-size:13px;color:#1E3448;flex:1;">{tier_desc}</div>
    </div>""", unsafe_allow_html=True)

    if not recs:
        st.markdown('<div class="icu-alert alert-ok" style="margin-top:20px;">✓ All risk scores below threshold — no recommendations triggered. Continue standard ICU monitoring protocols.</div>', unsafe_allow_html=True)
        return

    st.markdown('<div class="icu-section-label" style="margin-top:24px;">Clinical Recommendations</div>', unsafe_allow_html=True)
    by_cat = {}
    for cat,action,priority,color in recs: by_cat.setdefault(cat,[]).append((action,priority,color))

    p_cfg = {'STAT':('#6A1B9A','rgba(106,27,154,0.12)','🔴 STAT'),'URGENT':('#C62828','rgba(198,40,40,0.10)','🟠 URGENT'),'IMPORTANT':('#C8920A','rgba(200,146,10,0.08)','🟡 IMPORTANT'),'CONSIDER':('#1565C0','rgba(21,101,192,0.07)','🔵 CONSIDER')}
    cat_colors = {'Sepsis':('#C62828','🦠'),'Organ Failure':('#D4500A','🫘'),'Mortality':('#6A1B9A','💀'),'Criticality':('#1565C0','⚡')}

    for cat,items in by_cat.items():
        cat_color,cat_icon = cat_colors.get(cat,('#3A5F7A','📋'))
        sc = sum(1 for _,p,_ in items if p=='STAT'); uc = sum(1 for _,p,_ in items if p=='URGENT')
        counts_html = ''
        if sc: counts_html += f'<span style="background:rgba(106,27,154,0.15);color:#6A1B9A;font-family:JetBrains Mono,monospace;font-size:10px;font-weight:700;padding:2px 8px;border-radius:3px;margin-left:8px;">{sc} STAT</span>'
        if uc: counts_html += f'<span style="background:rgba(198,40,40,0.12);color:#C62828;font-family:JetBrains Mono,monospace;font-size:10px;font-weight:700;padding:2px 8px;border-radius:3px;margin-left:6px;">{uc} URGENT</span>'
        st.markdown(f"""
        <div style="display:flex;align-items:center;gap:10px;margin:20px 0 10px 0;padding:10px 14px;background:rgba(255,255,255,0.4);border-bottom:2px solid {cat_color};border-radius:6px 6px 0 0;">
            <span style="font-size:18px;">{cat_icon}</span>
            <span style="font-family:'Syne',sans-serif;font-size:15px;font-weight:700;color:{cat_color};">{cat}</span>
            <span style="font-family:'JetBrains Mono',monospace;font-size:10px;color:#3A5F7A;letter-spacing:0.1em;text-transform:uppercase;">— {len(items)} action{'s' if len(items)>1 else ''}</span>
            {counts_html}
        </div>""", unsafe_allow_html=True)
        for action,priority,color in items:
            fg,bg,label = p_cfg.get(priority,('#3A5F7A','rgba(0,0,0,0)',priority))
            st.markdown(f"""
            <div style="display:flex;align-items:stretch;gap:0;margin:4px 0;border-radius:6px;overflow:hidden;border:1px solid rgba(0,0,0,0.08);">
                <div style="background:{bg};border-right:3px solid {fg};padding:12px 14px;min-width:140px;display:flex;align-items:center;justify-content:center;flex-shrink:0;">
                    <span style="font-family:'JetBrains Mono',monospace;font-size:10px;font-weight:800;color:{fg};letter-spacing:0.08em;text-transform:uppercase;white-space:nowrap;">{label}</span>
                </div>
                <div style="background:rgba(255,255,255,0.4);padding:12px 16px;flex:1;font-family:'Syne',sans-serif;font-size:13px;font-weight:500;color:#1E3448;line-height:1.5;">{action}</div>
            </div>""", unsafe_allow_html=True)

    guidance = {'CRITICAL':('🚨 CRITICAL','All STAT recommendations require immediate bedside action. Notify attending and activate rapid response NOW.'),'SEVERE':('⚠️ SEVERE','Complete all URGENT recommendations within 1 hour. Reassess after each intervention.'),'HIGH':('🟡 HIGH','Elevated risk. Complete IMPORTANT recommendations within 2–4 hours this shift.'),'MODERATE':('📋 MODERATE','Moderate risk. Monitor closely and complete recommendations before end of shift.'),'LOW':('✓ LOW','Low risk profile. Continue standard ICU monitoring protocols.')}
    g_label,g_text = guidance.get(tier,('—',''))
    total_stat   = sum(1 for _,_,p,_ in recs if p=='STAT')
    total_urgent = sum(1 for _,_,p,_ in recs if p=='URGENT')
    sb = f'<div style="background:rgba(106,27,154,0.10);border:1px solid rgba(106,27,154,0.3);border-radius:6px;padding:8px 16px;text-align:center;"><div style="font-family:JetBrains Mono,monospace;font-size:20px;font-weight:700;color:#6A1B9A;">{total_stat}</div><div style="font-family:JetBrains Mono,monospace;font-size:9px;color:#3A5F7A;text-transform:uppercase;letter-spacing:0.1em;margin-top:2px;">STAT actions</div></div>' if total_stat else ''
    ub = f'<div style="background:rgba(198,40,40,0.08);border:1px solid rgba(198,40,40,0.3);border-radius:6px;padding:8px 16px;text-align:center;"><div style="font-family:JetBrains Mono,monospace;font-size:20px;font-weight:700;color:#C62828;">{total_urgent}</div><div style="font-family:JetBrains Mono,monospace;font-size:9px;color:#3A5F7A;text-transform:uppercase;letter-spacing:0.1em;margin-top:2px;">URGENT actions</div></div>' if total_urgent else ''
    st.markdown(f"""
    <div style="background:rgba(255,255,255,0.5);border:1px solid #90BDD8;border-left:4px solid {tc};border-radius:0 8px 8px 0;padding:18px 22px;margin-top:24px;">
        <div style="display:flex;justify-content:space-between;align-items:flex-start;flex-wrap:wrap;gap:12px;">
            <div>
                <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;letter-spacing:0.15em;text-transform:uppercase;margin-bottom:8px;">Clinical Context — {g_label}</div>
                <div style="font-family:'Syne',sans-serif;font-size:14px;font-weight:500;color:#0D1B2A;line-height:1.6;max-width:600px;">{g_text}</div>
            </div>
            <div style="display:flex;flex-direction:column;gap:6px;flex-shrink:0;">{sb}{ub}</div>
        </div>
        <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;margin-top:14px;padding-top:12px;border-top:1px solid #90BDD8;">
            ⚠ AI-generated — validate with qualified clinician before acting &nbsp;·&nbsp; MIMIC-IV research use only
        </div>
    </div>""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════
# TAB 8 — RLHF FEEDBACK
# ══════════════════════════════════════════════════
def tab_feedback(patient, doctor_id):
    sid = int(patient['stay_id'])
    st.markdown('<div class="icu-section-label">Doctor Feedback — RLHF Training</div>', unsafe_allow_html=True)
    st.markdown('<div style="font-family:\'Syne\',sans-serif;font-size:14px;color:#1E3448;margin-bottom:20px;line-height:1.6;">Your feedback improves the AI model through reinforcement learning from human feedback (RLHF). All responses are stored on Google Drive for model retraining.</div>', unsafe_allow_html=True)

    col_f, col_h = st.columns([3,2])
    with col_f:
        with st.form(f'fb_{sid}', clear_on_submit=True):
            st.markdown(f'<div style="font-family:\'JetBrains Mono\',monospace;font-size:11px;color:#3A5F7A;margin-bottom:18px;letter-spacing:0.1em;">FEEDBACK FOR STAY ID: {sid}</div>', unsafe_allow_html=True)
            agreed   = st.radio('Do AI predictions match your clinical assessment?', ['Yes — fully agree','Partially agree','Disagree','Insufficient data to judge'])
            useful   = st.radio('Were the recommendations clinically useful?', ['Very useful','Somewhat useful','Not useful','Not applicable'])
            severity = st.slider('Your severity assessment (0=stable, 10=critical)', 0, 10, 5)
            flag     = st.checkbox('🚩 Flag for model review team')
            notes    = st.text_area('Clinical notes / corrections (optional)', placeholder='Observations, corrections...', height=100)
            if st.form_submit_button('Submit Feedback', type='primary'):
                save_feedback({'timestamp':datetime.datetime.now().isoformat(),'stay_id':sid,'doctor_id':doctor_id or 'anonymous','prediction_agreed':agreed,'recommendation_useful':useful,'severity_rating':severity,'free_text':notes,'flag_for_review':flag,'pred_sepsis_prob':float(patient.get('pred_sepsis_prob',0) or 0),'pred_mortality_prob':float(patient.get('pred_mortality_prob',0) or 0),'pred_criticality_tier':str(patient.get('pred_criticality_tier','N/A') or 'N/A')})
                st.success('✅ Feedback saved!')

    with col_h:
        fb = load_feedback()
        st.markdown('<div class="icu-section-label">Feedback History</div>', unsafe_allow_html=True)
        if len(fb)==0: st.info('No feedback submitted yet.')
        else:
            total   = len(fb)
            agreed  = len(fb[fb['prediction_agreed'].str.startswith('Yes',na=False)])
            flagged = int(fb.get('flag_for_review',pd.Series([False]*total)).sum())
            st.markdown(f"""
            <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:10px;margin-bottom:18px;">
                <div class="icu-kpi" style="--accent-color:#1565C0;"><div class="icu-kpi-label">Total</div><div class="icu-kpi-value" style="font-size:24px;">{total}</div></div>
                <div class="icu-kpi" style="--accent-color:#1B8839;"><div class="icu-kpi-label">Agreement</div><div class="icu-kpi-value" style="color:#1B8839;font-size:24px;">{agreed/total*100:.0f}%</div></div>
                <div class="icu-kpi" style="--accent-color:#C62828;"><div class="icu-kpi-label">Flagged</div><div class="icu-kpi-value" style="color:#C62828;font-size:24px;">{flagged}</div></div>
            </div>""", unsafe_allow_html=True)
            if 'severity_rating' in fb.columns and len(fb)>1:
                fig = go.Figure(go.Histogram(x=fb['severity_rating'],nbinsx=11,marker=dict(color='#B5D8EC',line=dict(color='#1565C0',width=1))))
                fig.update_layout(title=dict(text='Severity Ratings Distribution',font=dict(size=14,color='#0D1B2A',family='Syne')),height=200,xaxis=dict(title='Score',gridcolor='#90BDD8',tickfont=dict(color='#0D1B2A',size=12)),yaxis=dict(gridcolor='#90BDD8',tickfont=dict(color='#0D1B2A',size=12)),**PLOTLY_BASE)
                st.plotly_chart(fig, use_container_width=True)
            csv = fb.to_csv(index=False).encode('utf-8')
            st.download_button('⬇ Download All Feedback (CSV)', data=csv, file_name=f'icu_feedback_{datetime.date.today()}.csv', mime='text/csv')

# ══════════════════════════════════════════════════
# TAB 9 — VIRTUAL ICU SANDBOX  (UPGRADED)
# ══════════════════════════════════════════════════

# ── Scenario presets ──────────────────────────────
SANDBOX_PRESETS = {
    'Custom': {},
    '🔴 Septic Shock Bundle (SSC Hour-1)': {
        'abx_timing': 'Within 1 hour',
        'abx_source': 'Unknown / Empirical',
        'fluids_ml': 2000,
        'vasopressor': 'Norepinephrine',
        'vaso_dose': 0.15,
        'ventilation': 'None',
        'peep': 5, 'fio2': 40,
        'transfusion': False, 'hb_target': 7.0,
        'insulin': True, 'glucose_target': '140–180 mg/dL',
        'dialysis': False,
        'steroids': True,
        'prone': False,
        'dvt_ppx': True,
        'stress_ulcer': True,
        'goals_of_care': False,
        'scenario_b': False,
    },
    '🫁 ARDS Protocol (Berlin)': {
        'abx_timing': 'None',
        'abx_source': 'Unknown / Empirical',
        'fluids_ml': 500,
        'vasopressor': 'None',
        'vaso_dose': 0.05,
        'ventilation': 'Invasive mechanical ventilation',
        'peep': 12, 'fio2': 60,
        'transfusion': False, 'hb_target': 7.0,
        'insulin': True, 'glucose_target': '140–180 mg/dL',
        'dialysis': False,
        'steroids': True,
        'prone': True,
        'dvt_ppx': True,
        'stress_ulcer': True,
        'goals_of_care': False,
        'scenario_b': False,
    },
    '🫘 AKI / Oliguric Protocol': {
        'abx_timing': 'None',
        'abx_source': 'Unknown / Empirical',
        'fluids_ml': 1000,
        'vasopressor': 'Norepinephrine',
        'vaso_dose': 0.08,
        'ventilation': 'None',
        'peep': 5, 'fio2': 40,
        'transfusion': False, 'hb_target': 8.0,
        'insulin': True, 'glucose_target': '140–180 mg/dL',
        'dialysis': True,
        'steroids': False,
        'prone': False,
        'dvt_ppx': True,
        'stress_ulcer': True,
        'goals_of_care': False,
        'scenario_b': False,
    },
    '💀 Palliative / Goals of Care': {
        'abx_timing': 'None',
        'abx_source': 'Unknown / Empirical',
        'fluids_ml': 0,
        'vasopressor': 'None',
        'vaso_dose': 0.05,
        'ventilation': 'None',
        'peep': 5, 'fio2': 40,
        'transfusion': False, 'hb_target': 7.0,
        'insulin': False, 'glucose_target': '140–180 mg/dL',
        'dialysis': False,
        'steroids': False,
        'prone': False,
        'dvt_ppx': False,
        'stress_ulcer': False,
        'goals_of_care': True,
        'scenario_b': False,
    },
}

def apply_intervention_effects(base_probs, interventions):
    """
    Evidence-informed rule engine with interaction effects,
    contraindication warnings, and reassessment checkpoints.
    Returns: updated probs dict, timeline list, warnings list,
             reassessment checklist dict, P/F ratio estimate.
    """
    sep  = base_probs['sepsis']
    org  = base_probs['organ']
    mort = base_probs['mortality']
    timeline = []
    warnings_out = []

    # ── Track which interventions are active for interaction effects ──
    has_abx   = interventions.get('abx_timing','None') != 'None'
    has_fluid = interventions.get('fluids_ml', 0) > 0
    has_vaso  = interventions.get('vasopressor','None') != 'None'
    has_vent  = interventions.get('ventilation','None') != 'None'
    has_inv   = interventions.get('ventilation','None') == 'Invasive mechanical ventilation'
    fl        = interventions.get('fluids_ml', 0)
    vaso_dose = interventions.get('vaso_dose', 0.1)

    # ── 1. ANTIBIOTICS ────────────────────────────────────────────────
    abx = interventions.get('abx_timing','None')
    abx_src = interventions.get('abx_source','Unknown / Empirical')
    if abx == 'Within 1 hour':
        sep *= 0.52; mort *= 0.72
        timeline += [
            ('0h','#1B8839','Blood cultures ×2 drawn BEFORE antibiotics'),
            ('0h','#1B8839',f'Broad-spectrum IV antibiotics started — source: {abx_src}'),
            ('1h','#1B8839','Lactate repeat ordered — Sepsis bundle initiated'),
        ]
    elif abx == 'Within 3 hours':
        sep *= 0.63; mort *= 0.80
        timeline += [
            ('0h','#C8920A','Cultures drawn — antibiotics ordered'),
            ('3h','#C8920A',f'Antibiotics administered — source: {abx_src}'),
        ]
    elif abx == 'Within 6 hours':
        sep *= 0.76; mort *= 0.89
        timeline += [
            ('0h','#D4500A','Antibiotics delayed — cultures drawn'),
            ('6h','#D4500A',f'Antibiotics administered (delayed) — source: {abx_src}'),
        ]
        warnings_out.append(('⚠️ DELAY WARNING', 'Antibiotic delay >3h significantly worsens sepsis mortality. Expedite.', '#C62828'))

    # ── 2. IV FLUIDS ──────────────────────────────────────────────────
    if fl > 0:
        # Interaction: fluids + vasopressor together is better than either alone
        fluid_vaso_bonus = 0.94 if has_vaso else 1.0
        if fl <= 500:
            org *= 0.97; mort *= 0.98
            timeline.append(('0h','#1565C0',f'{fl} mL conservative IV crystalloid bolus'))
        elif fl <= 1500:
            org *= 0.91 * fluid_vaso_bonus; mort *= 0.93 * fluid_vaso_bonus; sep *= 0.92
            timeline += [
                ('0h','#1565C0',f'{fl} mL IV crystalloid resuscitation started'),
                ('1h','#1565C0','MAP & urine output response assessed — target MAP > 65 mmHg'),
            ]
        elif fl <= 3000:
            org *= 0.86 * fluid_vaso_bonus; mort *= 0.87 * fluid_vaso_bonus; sep *= 0.86
            timeline += [
                ('0h','#1565C0',f'{fl} mL aggressive fluid resuscitation (30 mL/kg protocol)'),
                ('1h','#1565C0','Passive leg raise — assess fluid responsiveness'),
                ('2h','#C8920A','Reassess for fluid overload — auscultate chest'),
            ]
        else:
            # Excessive fluids — penalise organ failure
            org *= 0.89 * fluid_vaso_bonus; mort *= 0.87 * fluid_vaso_bonus; sep *= 0.85
            org *= 1.10  # fluid overload penalty
            timeline += [
                ('0h','#C62828',f'{fl} mL — CAUTION: high-volume resuscitation'),
                ('2h','#C62828','Monitor for pulmonary oedema — CXR recommended'),
                ('3h','#C8920A','Consider albumin or colloid if crystalloid > 3L'),
            ]
            warnings_out.append(('⚠️ FLUID OVERLOAD RISK', f'{fl} mL crystalloid exceeds 30 mL/kg for average patient. Monitor respiratory status closely.', '#C62828'))

    # ── 3. VASOPRESSORS ───────────────────────────────────────────────
    vp = interventions.get('vasopressor','None')
    # Dose-response: higher dose = more benefit but more arrhythmia risk
    dose_factor = min(vaso_dose / 0.10, 1.5)  # normalised around 0.10 mcg/kg/min
    if vp == 'Norepinephrine':
        org *= max(0.82, 0.88 - 0.04 * dose_factor); mort *= max(0.78, 0.84 - 0.04 * dose_factor)
        timeline += [
            ('0h','#6A1B9A',f'Norepinephrine {vaso_dose:.2f} mcg/kg/min — central line confirmed'),
            ('15m','#6A1B9A','MAP target > 65 mmHg — titrate every 15 min'),
            ('1h','#6A1B9A','ScvO₂ target > 70% — reassess cardiac output'),
        ]
        if vaso_dose > 0.25:
            warnings_out.append(('⚠️ HIGH VASOPRESSOR DOSE', f'Norepinephrine {vaso_dose:.2f} mcg/kg/min is high. Consider adding vasopressin to spare dose.', '#D4500A'))
    elif vp == 'Vasopressin':
        org *= 0.90; mort *= 0.86
        timeline += [
            ('0h','#6A1B9A','Vasopressin 0.03 units/min added as norepinephrine-sparing agent'),
            ('30m','#6A1B9A','Monitor for hyponatraemia and mesenteric ischaemia'),
        ]
    elif vp == 'Dopamine':
        org *= 0.91; mort *= 0.89
        timeline += [
            ('0h','#6A1B9A',f'Dopamine {vaso_dose*10:.0f} mcg/kg/min infusion started'),
            ('30m','#C8920A','Cardiac monitoring — dopamine increases arrhythmia risk'),
            ('1h','#C8920A','Consider switching to norepinephrine if haemodynamically unstable'),
        ]
        warnings_out.append(('ℹ️ DOPAMINE CAUTION', 'SSC 2021 guidelines recommend norepinephrine over dopamine as first-line vasopressor.', '#C8920A'))
    elif vp == 'Epinephrine':
        org *= 0.87; mort *= 0.83; sep *= 0.88
        timeline += [
            ('0h','#C62828','Epinephrine started — reserved for refractory shock'),
            ('15m','#C62828','Monitor lactate — epinephrine can elevate lactate artifactually'),
        ]
        warnings_out.append(('⚠️ EPINEPHRINE ALERT', 'Epinephrine may cause lactic acidosis masking true resuscitation response. Monitor ScvO₂ and clinical parameters.', '#C62828'))

    # ── 4. VENTILATION + PARAMETERS ──────────────────────────────────
    vent   = interventions.get('ventilation','None')
    peep   = interventions.get('peep', 5)
    fio2   = interventions.get('fio2', 40)
    # Estimated PaO2 from FiO2 (simplified: PaO2 ≈ FiO2 * 5 in sick lungs)
    pao2_est  = fio2 * 3.5
    pf_ratio  = round(pao2_est / (fio2 / 100), 0) if fio2 > 0 else 300
    # PEEP benefit on oxygenation
    peep_bonus = max(0.90, 1.0 - (peep - 5) * 0.012)  # higher PEEP → better oxygenation

    if vent == 'High-flow O₂ (HFNC)':
        org *= 0.88 * peep_bonus; mort *= 0.91
        timeline += [
            ('0h','#0097A7',f'HFNC initiated — FiO₂ {fio2}%, flow 40–60 L/min'),
            ('1h','#0097A7','SpO₂ target ≥ 92% — reassess work of breathing'),
            ('2h','#C8920A','If ROX index < 4.88 at 2h — consider escalation to NIV/intubation'),
        ]
    elif vent == 'Non-invasive (NIV/BiPAP)':
        org *= 0.80 * peep_bonus; mort *= 0.84
        timeline += [
            ('0h','#0097A7',f'NIV/BiPAP initiated — PEEP {peep} cmH₂O, FiO₂ {fio2}%'),
            ('1h','#0097A7','ABG at 1h — assess ventilatory response (pH, PaCO₂, PaO₂)'),
            ('2h','#0097A7','SpO₂ target ≥ 92%, RR < 25/min — reassess for intubation need'),
        ]
        if peep > 10:
            warnings_out.append(('ℹ️ HIGH PEEP (NIV)', f'PEEP {peep} cmH₂O on NIV — ensure tight mask seal and monitor haemodynamics.', '#C8920A'))
    elif vent == 'Invasive mechanical ventilation':
        org *= 0.72 * peep_bonus; mort *= 0.77
        # Interaction: IMV + PEEP > 8 benefits ARDS patients
        if peep >= 8:
            org  *= 0.94; mort *= 0.95
        timeline += [
            ('0h','#C62828','RSI intubation — confirm ETT position (CXR + capnography)'),
            ('0h','#0097A7',f'ARDSnet: TV 6 mL/kg IBW, PEEP {peep} cmH₂O, FiO₂ {fio2}%'),
            ('0h','#0097A7','Plateau pressure target < 30 cmH₂O — check compliance'),
            ('1h','#0097A7','Sedation/analgesia: propofol + fentanyl — RASS target -1 to -2'),
            ('4h','#1B8839','Spontaneous awakening trial protocol initiated'),
            ('6h','#1B8839','Daily SBT — assess readiness for extubation'),
        ]
        if peep > 15:
            warnings_out.append(('⚠️ HIGH PEEP WARNING', f'PEEP {peep} cmH₂O may impair venous return and cardiac output. Monitor haemodynamics closely.', '#C62828'))

    # ── 5. PRONE POSITIONING ─────────────────────────────────────────
    if interventions.get('prone', False):
        if has_inv:
            # Prone only beneficial with IMV + severe ARDS (P/F < 150)
            if pf_ratio < 150:
                org *= 0.82; mort *= 0.78
                timeline += [
                    ('0h','#6A1B9A','Prone positioning initiated — P/F < 150, severe ARDS criteria met'),
                    ('1h','#6A1B9A','16-hour prone session — monitor pressure areas and ETT'),
                    ('16h','#6A1B9A','Supine turn — reassess P/F ratio for repeat proning'),
                ]
            else:
                org *= 0.93; mort *= 0.95
                timeline.append(('0h','#C8920A','Prone initiated — consider P/F < 150 threshold for maximum benefit'))
                warnings_out.append(('ℹ️ PRONE THRESHOLD', f'P/F ratio estimated {pf_ratio:.0f}. Prone positioning has greatest benefit when P/F < 150 (Berlin criteria for severe ARDS).', '#C8920A'))
        else:
            warnings_out.append(('⚠️ PRONE REQUIRES IMV', 'Prone positioning is indicated only with invasive mechanical ventilation. Select IMV first.', '#C62828'))

    # ── 6. RENAL REPLACEMENT ─────────────────────────────────────────
    if interventions.get('dialysis', False):
        org *= 0.77; mort *= 0.84
        timeline += [
            ('0h','#D4500A','CRRT initiated — KDIGO AKI Stage ≥ 2 criteria'),
            ('1h','#D4500A','Anticoagulation: citrate or heparin — check contraindications'),
            ('4h','#1B8839','Fluid balance improving — net negative balance target'),
            ('12h','#1B8839','Electrolytes: K⁺, Ca²⁺, Mg²⁺ — correct replacement'),
        ]

    # ── 7. CORTICOSTEROIDS ───────────────────────────────────────────
    if interventions.get('steroids', False):
        # Benefit is amplified in septic shock with vasopressors
        steroid_bonus = 0.90 if has_vaso else 0.95
        sep *= 0.80 * steroid_bonus; mort *= 0.86 * steroid_bonus
        timeline += [
            ('0h','#C8920A','Hydrocortisone 200 mg/day (50 mg q6h IV) — septic shock refractory'),
            ('24h','#C8920A','Reassess vasopressor requirement — wean if MAP stable'),
            ('7d','#C8920A','Taper steroids over 7 days — avoid abrupt cessation'),
        ]
        if not has_vaso:
            warnings_out.append(('ℹ️ STEROID INDICATION', 'SSC 2021: steroids recommended when vasopressors are required AND patient remains haemodynamically unstable. Consider vasopressor first.', '#C8920A'))

    # ── 8. TRANSFUSION ───────────────────────────────────────────────
    if interventions.get('transfusion', False):
        hb_tgt = interventions.get('hb_target', 7.0)
        if hb_tgt <= 7.0:
            org *= 0.96; mort *= 0.96
            timeline += [
                ('0h','#C62828',f'pRBC transfusion — restrictive strategy (Hb target ≥ {hb_tgt} g/dL)'),
                ('2h','#1B8839','Post-transfusion Hb — confirm therapeutic response'),
            ]
        elif hb_tgt <= 8.0:
            org *= 0.94; mort *= 0.94
            timeline += [
                ('0h','#C8920A',f'pRBC transfusion — Hb target ≥ {hb_tgt} g/dL (ACS/cardiac protocol)'),
                ('2h','#1B8839','Post-transfusion Hb — confirm therapeutic response'),
            ]
        else:
            org *= 0.95; mort *= 0.97
            org *= 1.03  # liberal strategy slight organ penalty
            timeline += [
                ('0h','#C8920A',f'pRBC transfusion — liberal strategy Hb target {hb_tgt} g/dL'),
            ]
            warnings_out.append(('ℹ️ LIBERAL TRANSFUSION', f'Hb target {hb_tgt} g/dL is above restrictive threshold. TRICC/TRISS trials support restrictive strategy (7 g/dL) in most ICU patients.', '#C8920A'))

    # ── 9. INSULIN / GLUCOSE CONTROL ─────────────────────────────────
    if interventions.get('insulin', False):
        g_target = interventions.get('glucose_target','140–180 mg/dL')
        if g_target == '80–110 mg/dL (Tight)':
            org *= 0.93; mort *= 0.94
            warnings_out.append(('⚠️ TIGHT GLUCOSE CAUTION', 'Tight glycaemic control (80–110 mg/dL) increases hypoglycaemia risk (NICE-SUGAR trial). Target 140–180 mg/dL is safer in most ICU patients.', '#C62828'))
            timeline.append(('0h','#C8920A',f'Insulin infusion — tight target {g_target} — frequent glucose monitoring q1h'))
        else:
            org *= 0.95; mort *= 0.96
            timeline.append(('0h','#1B8839',f'Insulin infusion — target {g_target} — glucose monitoring q2h'))

    # ── 10. DVT PROPHYLAXIS ──────────────────────────────────────────
    if interventions.get('dvt_ppx', False):
        mort *= 0.97
        timeline.append(('0h','#1565C0','VTE prophylaxis: LMWH + SCDs — contraindications checked'))

    # ── 11. STRESS ULCER PROPHYLAXIS ────────────────────────────────
    if interventions.get('stress_ulcer', False):
        mort *= 0.98
        timeline.append(('0h','#1565C0','Stress ulcer prophylaxis: pantoprazole 40 mg IV daily'))

    # ── 12. GOALS OF CARE ───────────────────────────────────────────
    if interventions.get('goals_of_care', False):
        timeline += [
            ('0h','#3A5F7A','Goals of care discussion initiated with patient/family/surrogate'),
            ('2h','#3A5F7A','Palliative care and ethics teams consulted'),
            ('4h','#3A5F7A','Advance directive / code status confirmed and documented'),
        ]

    # ── INTERACTION EFFECTS ──────────────────────────────────────────
    # Full bundle (abx + fluid + vaso) = synergistic bonus
    if has_abx and has_fluid and has_vaso:
        sep *= 0.90; mort *= 0.88
        timeline.append(('0h','#1B8839','✅ Full Surviving Sepsis Hour-1 Bundle achieved — synergistic benefit'))

    # IMV without vasopressor in haemodynamically unstable patient
    if has_inv and not has_vaso and base_probs['organ'] > 0.4:
        org *= 1.06  # intubation sedation → haemodynamic instability
        warnings_out.append(('⚠️ INTUBATION + NO VASOPRESSOR', 'Induction agents cause vasodilation. Have vasopressor running BEFORE intubation in haemodynamically unstable patients.', '#C62828'))

    # ── CLAMP & SORT ─────────────────────────────────────────────────
    sep  = max(0.01, min(0.99, sep))
    org  = max(0.01, min(0.99, org))
    mort = max(0.01, min(0.99, mort))

    order = {'0h':0,'15m':1,'30m':2,'1h':3,'2h':4,'3h':5,'4h':6,'6h':7,'7d':8,'12h':9,'16h':10,'24h':11}
    timeline.sort(key=lambda x: order.get(x[0], 99))

    # ── REASSESSMENT CHECKPOINTS ─────────────────────────────────────
    reassess = {
        '1h': [],
        '3h': [],
        '6h': [],
        '24h': [],
    }
    if has_abx:
        reassess['1h'].append('Lactate repeat — target < 2 mmol/L or ≥ 10% clearance')
        reassess['3h'].append('Culture results — de-escalate antibiotics if sensitivities available')
    if has_fluid or has_vaso:
        reassess['1h'].append('MAP response — target > 65 mmHg')
        reassess['1h'].append('Urine output — target > 0.5 mL/kg/hr')
        reassess['3h'].append('ScvO₂ — target > 70%')
        reassess['6h'].append('Fluid balance — reassess need for further resuscitation')
    if has_inv:
        reassess['1h'].append('ABG — pH, PaO₂, PaCO₂, plateau pressure < 30 cmH₂O')
        reassess['6h'].append('Spontaneous awakening trial — assess sedation level (RASS)')
        reassess['24h'].append('Daily SBT — extubation readiness assessment')
    if interventions.get('dialysis', False):
        reassess['6h'].append('Electrolytes (K⁺, Ca²⁺, Mg²⁺) — CRRT replacement solution')
        reassess['24h'].append('Fluid balance review — adjust CRRT ultrafiltration rate')
    if interventions.get('steroids', False):
        reassess['24h'].append('Vasopressor dose — wean hydrocortisone if MAP stable')
    if interventions.get('insulin', False):
        reassess['1h'].append('Blood glucose — adjust insulin infusion rate')

    return (
        {'sepsis': sep, 'organ': org, 'mortality': mort},
        timeline,
        warnings_out,
        reassess,
        pf_ratio,
    )


def _render_intervention_block(title, desc, icon=''):
    st.markdown(
        f'<div class="sandbox-card"><div class="sandbox-title">{icon} {title}</div>'
        f'<div class="sandbox-desc">{desc}</div></div>',
        unsafe_allow_html=True
    )


def tab_sandbox(patient):
    sep_base  = float(patient.get('pred_sepsis_prob', 0)        or 0)
    org_base  = float(patient.get('pred_organ_failure_prob', 0) or 0)
    mort_base = float(patient.get('pred_mortality_prob', 0)     or 0)
    tier = str(patient.get('pred_criticality_tier','N/A') or 'N/A')
    tc   = TIER_COLORS.get(tier,'#3A5F7A')
    base_probs = {'sepsis': sep_base, 'organ': org_base, 'mortality': mort_base}

    # ── Header ────────────────────────────────────────────────────────
    st.markdown("""
    <div style="background:linear-gradient(135deg,rgba(21,101,192,0.06) 0%,rgba(0,151,167,0.04) 100%);
                border:1px solid rgba(21,101,192,0.2);border-radius:10px;padding:16px 20px;margin-bottom:16px;">
        <div style="font-family:'Syne',sans-serif;font-size:18px;font-weight:800;color:#0D1B2A;margin-bottom:4px;">
            🧪 Virtual ICU Sandbox
        </div>
        <div style="font-family:'JetBrains Mono',monospace;font-size:11px;color:#3A5F7A;line-height:1.7;">
            Simulate evidence-based interventions · Interaction effects modelled ·
            Reassessment checkpoints · Side-by-side scenario comparison ·
            Handover export · Educational use only
        </div>
    </div>""", unsafe_allow_html=True)

    # ── Current patient state ─────────────────────────────────────────
    st.markdown('<div class="icu-section-label">Current Patient Baseline</div>', unsafe_allow_html=True)
    def _slabel(v):
        if v > 0.75: return 'CRITICAL'
        if v > 0.50: return 'SEVERE'
        if v > 0.25: return 'TRIGGERED'
        if v > 0.10: return 'ELEVATED'
        return 'NORMAL'
    cs1,cs2,cs3,cs4 = st.columns(4)
    for col,label,val,color in [
        (cs1,'Sepsis Risk',   _slabel(sep_base),  TIER_COLORS['SEVERE' if sep_base>0.5  else 'MODERATE' if sep_base>0.25  else 'LOW']),
        (cs2,'Organ Failure', _slabel(org_base),  TIER_COLORS['SEVERE' if org_base>0.5  else 'MODERATE' if org_base>0.25  else 'LOW']),
        (cs3,'Mortality Risk',_slabel(mort_base), TIER_COLORS['SEVERE' if mort_base>0.5 else 'MODERATE' if mort_base>0.25 else 'LOW']),
        (cs4,'Risk Tier',     tier,               tc),
    ]:
        with col:
            st.markdown(
                f'<div class="icu-kpi" style="--accent-color:{color};">'
                f'<div class="icu-kpi-label">{label}</div>'
                f'<div class="icu-kpi-value" style="color:{color};font-size:22px;">{val}</div>'
                f'<div class="icu-kpi-sub">Baseline — no intervention</div></div>',
                unsafe_allow_html=True
            )

    # ── Scenario preset picker ────────────────────────────────────────
    st.markdown('<div class="icu-section-label">Scenario Preset</div>', unsafe_allow_html=True)
    preset_choice = st.selectbox(
        'Load a clinical scenario preset or configure manually below',
        list(SANDBOX_PRESETS.keys()), key='sandbox_preset'
    )
    preset = SANDBOX_PRESETS[preset_choice]

    def _pv(key, default):
        """Return preset value if preset is active, else default."""
        return preset.get(key, default) if preset_choice != 'Custom' else default

    st.markdown('<div class="icu-section-label">Configure Interventions</div>', unsafe_allow_html=True)

    # ── Two-column intervention grid ──────────────────────────────────
    col_L, col_R = st.columns(2)

    with col_L:
        # 1. Antibiotics
        _render_intervention_block('Antibiotic Therapy', 'SEPSIS-3 · HOUR-1 BUNDLE', '💊')
        _abx_opts = ['None','Within 1 hour','Within 3 hours','Within 6 hours']
        _abx_val  = _pv('abx_timing', 'None')
        _abx_idx  = _abx_opts.index(_abx_val) if _abx_val in _abx_opts else 0
        abx_timing = st.selectbox(
            'Time to antibiotics',
            _abx_opts,
            index=_abx_idx,
            key='sandbox_abx'
        )
        _abx_src_opts = ['Unknown / Empirical','Pulmonary','Abdominal','Urinary',
                         'Line / CRBSI','Skin / Soft Tissue','CNS / Meningitis','Endocarditis']
        _abx_src_val  = _pv('abx_source', 'Unknown / Empirical')
        _abx_src_idx  = _abx_src_opts.index(_abx_src_val) if _abx_src_val in _abx_src_opts else 0
        abx_source = st.selectbox(
            'Suspected infection source',
            _abx_src_opts,
            index=_abx_src_idx,
            key='sandbox_abx_src'
        )

        st.markdown('<div style="height:8px;"></div>', unsafe_allow_html=True)

        # 2. Fluids
        _render_intervention_block('IV Fluid Resuscitation', 'CRYSTALLOID · 30 mL/kg PROTOCOL', '💧')
        fluids_ml = st.slider(
            'Total IV crystalloid (mL)', 0, 5000,
            value=_pv('fluids_ml', 0), step=250, key='sandbox_fluids'
        )
        if fluids_ml > 0:
            ibw_kg = 70  # assumed IBW
            per_kg = fluids_ml / ibw_kg
            fl_color = '#C62828' if per_kg > 35 else '#C8920A' if per_kg > 25 else '#1B8839'
            st.markdown(
                f'<div style="font-family:JetBrains Mono,monospace;font-size:11px;'
                f'color:{fl_color};margin-top:4px;">'
                f'≈ {per_kg:.0f} mL/kg (IBW 70 kg) — '
                f'{"⚠️ exceeds 30 mL/kg" if per_kg>30 else "within 30 mL/kg protocol"}</div>',
                unsafe_allow_html=True
            )

        st.markdown('<div style="height:8px;"></div>', unsafe_allow_html=True)

        # 3. Vasopressors
        _render_intervention_block('Vasopressor Therapy', 'MAP TARGET > 65 mmHg · TITRATION', '💉')
        _vaso_opts = ['None','Norepinephrine','Vasopressin','Dopamine','Epinephrine']
        _vaso_val  = _pv('vasopressor', 'None')
        _vaso_idx  = _vaso_opts.index(_vaso_val) if _vaso_val in _vaso_opts else 0
        vasopressor = st.selectbox(
            'Vasopressor agent',
            _vaso_opts,
            index=_vaso_idx,
            key='sandbox_vaso'
        )
        vaso_dose = 0.05
        if vasopressor != 'None':
            vaso_dose = st.slider(
                f'{vasopressor} dose (mcg/kg/min)', 0.01, 0.50,
                value=float(_pv('vaso_dose', 0.10)), step=0.01,
                key='sandbox_vaso_dose'
            )
            dose_color = '#C62828' if vaso_dose > 0.25 else '#C8920A' if vaso_dose > 0.15 else '#1B8839'
            dose_label = 'HIGH' if vaso_dose > 0.25 else 'MODERATE' if vaso_dose > 0.15 else 'LOW'
            st.markdown(
                f'<div style="font-family:JetBrains Mono,monospace;font-size:11px;'
                f'color:{dose_color};margin-top:4px;">Dose level: {dose_label}</div>',
                unsafe_allow_html=True
            )

        st.markdown('<div style="height:8px;"></div>', unsafe_allow_html=True)

        # 4. Transfusion
        _render_intervention_block('Blood Transfusion', 'pRBC · RESTRICTIVE VS LIBERAL', '🩸')
        transfusion = st.checkbox(
            'Administer packed red blood cells (pRBC)',
            value=_pv('transfusion', False), key='sandbox_transfusion'
        )
        hb_target = 7.0
        if transfusion:
            hb_target = st.select_slider(
                'Hb trigger / target (g/dL)',
                options=[7.0, 7.5, 8.0, 8.5, 9.0, 10.0],
                value=float(_pv('hb_target', 7.0)),
                key='sandbox_hb'
            )

    with col_R:
        # 5. Ventilation
        _render_intervention_block('Respiratory Support', 'OXYGENATION STRATEGY · ARDSnet', '🫁')
        _vent_opts = ['None','High-flow O₂ (HFNC)','Non-invasive (NIV/BiPAP)','Invasive mechanical ventilation']
        _vent_val  = _pv('ventilation', 'None')
        _vent_idx  = _vent_opts.index(_vent_val) if _vent_val in _vent_opts else 0
        ventilation = st.selectbox(
            'Ventilation mode',
            _vent_opts,
            index=_vent_idx,
            key='sandbox_vent'
        )
        peep, fio2 = 5, 40
        if ventilation != 'None':
            vc1, vc2 = st.columns(2)
            with vc1:
                peep = st.slider('PEEP (cmH₂O)', 0, 20, value=int(_pv('peep', 5)), key='sandbox_peep')
            with vc2:
                fio2 = st.slider('FiO₂ (%)', 21, 100, value=int(_pv('fio2', 40)), key='sandbox_fio2')
            pao2_est = fio2 * 3.5
            pf_disp  = round(pao2_est / (fio2/100), 0)
            pf_color = '#C62828' if pf_disp < 100 else '#D4500A' if pf_disp < 200 else '#C8920A' if pf_disp < 300 else '#1B8839'
            pf_label = 'SEVERE ARDS' if pf_disp < 100 else 'MODERATE ARDS' if pf_disp < 200 else 'MILD ARDS' if pf_disp < 300 else 'NORMAL'
            st.markdown(
                f'<div style="font-family:JetBrains Mono,monospace;font-size:11px;'
                f'color:{pf_color};margin-top:4px;">Est P/F ratio: {pf_disp:.0f} — {pf_label}</div>',
                unsafe_allow_html=True
            )

        st.markdown('<div style="height:8px;"></div>', unsafe_allow_html=True)

        # 6. Prone positioning
        _render_intervention_block('Prone Positioning', 'SEVERE ARDS · P/F < 150 · 16h SESSIONS', '🔄')
        prone = st.checkbox(
            'Initiate prone positioning (requires IMV)',
            value=_pv('prone', False), key='sandbox_prone'
        )

        st.markdown('<div style="height:8px;"></div>', unsafe_allow_html=True)

        # 7. Renal replacement
        _render_intervention_block('Renal Replacement Therapy', 'CRRT · AKI STAGE ≥ 2', '🫘')
        dialysis = st.checkbox(
            'Initiate CRRT / dialysis',
            value=_pv('dialysis', False), key='sandbox_dialysis'
        )

        st.markdown('<div style="height:8px;"></div>', unsafe_allow_html=True)

        # 8. Adjunctive therapies
        _render_intervention_block('Adjunctive Therapies', 'STEROIDS · INSULIN · PROPHYLAXIS', '💊')
        steroids = st.checkbox(
            'Hydrocortisone 200 mg/day (refractory septic shock)',
            value=_pv('steroids', False), key='sandbox_steroids'
        )
        insulin = st.checkbox(
            'Insulin infusion (glycaemic control)',
            value=_pv('insulin', False), key='sandbox_insulin'
        )
        glucose_target = '140–180 mg/dL'
        if insulin:
            glucose_target = st.selectbox(
                'Glucose target',
                ['140–180 mg/dL','110–140 mg/dL','80–110 mg/dL (Tight)'],
                index=['140–180 mg/dL','110–140 mg/dL','80–110 mg/dL (Tight)'].index(_pv('glucose_target','140–180 mg/dL')),
                key='sandbox_glucose'
            )
        dvt_ppx     = st.checkbox('DVT prophylaxis (LMWH + SCDs)',        value=_pv('dvt_ppx', False),     key='sandbox_dvt')
        stress_ulcer= st.checkbox('Stress ulcer prophylaxis (PPI)',        value=_pv('stress_ulcer', False), key='sandbox_sux')
        goc         = st.checkbox('Initiate goals of care discussion',      value=_pv('goals_of_care', False),key='sandbox_goc')

    # ── Scenario B toggle (comparison mode) ──────────────────────────
    st.markdown('<div class="icu-section-label">Scenario Comparison</div>', unsafe_allow_html=True)
    scenario_b = st.checkbox(
        '🔀 Enable Scenario B — compare two management strategies side by side',
        value=False, key='sandbox_scenario_b'
    )

    interventions_a = {
        'abx_timing': abx_timing, 'abx_source': abx_source,
        'fluids_ml': fluids_ml, 'vasopressor': vasopressor, 'vaso_dose': vaso_dose,
        'ventilation': ventilation, 'peep': peep, 'fio2': fio2,
        'prone': prone, 'dialysis': dialysis, 'steroids': steroids,
        'insulin': insulin, 'glucose_target': glucose_target,
        'transfusion': transfusion, 'hb_target': hb_target,
        'dvt_ppx': dvt_ppx, 'stress_ulcer': stress_ulcer,
        'goals_of_care': goc,
    }

    interventions_b = {}
    if scenario_b:
        st.markdown("""
        <div style="background:rgba(0,151,167,0.06);border:1px solid rgba(0,151,167,0.2);
                    border-radius:8px;padding:12px 16px;margin-bottom:12px;">
            <div style="font-family:'Syne',sans-serif;font-size:13px;font-weight:700;color:#0097A7;">
                ⬇ Configure Scenario B
            </div>
        </div>""", unsafe_allow_html=True)
        sb1, sb2 = st.columns(2)
        with sb1:
            abx_b   = st.selectbox('Scenario B — Antibiotics', ['None','Within 1 hour','Within 3 hours','Within 6 hours'], key='sb_abx')
            fluid_b = st.slider('Scenario B — IV Fluid (mL)', 0, 5000, 0, step=250, key='sb_fluids')
            vaso_b  = st.selectbox('Scenario B — Vasopressor', ['None','Norepinephrine','Vasopressin','Dopamine','Epinephrine'], key='sb_vaso')
        with sb2:
            vent_b  = st.selectbox('Scenario B — Ventilation', ['None','High-flow O₂ (HFNC)','Non-invasive (NIV/BiPAP)','Invasive mechanical ventilation'], key='sb_vent')
            dial_b  = st.checkbox('Scenario B — CRRT', key='sb_dialysis')
            ster_b  = st.checkbox('Scenario B — Steroids', key='sb_steroids')
            goc_b   = st.checkbox('Scenario B — Goals of Care', key='sb_goc')
        interventions_b = {
            'abx_timing': abx_b, 'abx_source': 'Unknown / Empirical',
            'fluids_ml': fluid_b, 'vasopressor': vaso_b, 'vaso_dose': 0.10,
            'ventilation': vent_b, 'peep': 5, 'fio2': 40,
            'prone': False, 'dialysis': dial_b, 'steroids': ster_b,
            'insulin': False, 'glucose_target': '140–180 mg/dL',
            'transfusion': False, 'hb_target': 7.0,
            'dvt_ppx': False, 'stress_ulcer': False, 'goals_of_care': goc_b,
        }

    # ── Check any intervention selected ──────────────────────────────
    any_int = any([
        abx_timing != 'None', fluids_ml > 0, vasopressor != 'None',
        ventilation != 'None', prone, dialysis, steroids,
        insulin, transfusion, dvt_ppx, stress_ulcer, goc,
    ])

    st.markdown('<div class="icu-section-label">Simulation Results</div>', unsafe_allow_html=True)
    if not any_int:
        st.markdown(
            '<div class="icu-alert alert-info" style="text-align:center;padding:28px;">'
            '👆 Select one or more interventions above to run the simulation'
            '</div>',
            unsafe_allow_html=True
        )
        return

    # ── Run simulation ────────────────────────────────────────────────
    new_probs_a, timeline_a, warnings_a, reassess_a, pf_a = apply_intervention_effects(base_probs, interventions_a)

    new_probs_b, timeline_b, warnings_b, reassess_b, pf_b = ({}, [], [], {}, 300)
    if scenario_b and interventions_b:
        new_probs_b, timeline_b, warnings_b, reassess_b, pf_b = apply_intervention_effects(base_probs, interventions_b)

    # ── Clinical warnings ─────────────────────────────────────────────
    if warnings_a:
        st.markdown('<div class="icu-section-label">⚠️ Clinical Alerts</div>', unsafe_allow_html=True)
        for w_title, w_text, w_color in warnings_a:
            st.markdown(
                f'<div style="background:rgba(0,0,0,0.03);border-left:4px solid {w_color};'
                f'border-radius:0 6px 6px 0;padding:10px 16px;margin:6px 0;">'
                f'<div style="font-family:JetBrains Mono,monospace;font-size:10px;font-weight:700;'
                f'color:{w_color};letter-spacing:0.1em;">{w_title}</div>'
                f'<div style="font-family:Syne,sans-serif;font-size:13px;color:#1E3448;margin-top:4px;">{w_text}</div>'
                f'</div>',
                unsafe_allow_html=True
            )

    # ── Delta cards ───────────────────────────────────────────────────
    def _dc(before, after):
        d = after - before
        if d < -0.15: return '#1B8839','#D4EDDA'
        if d < -0.05: return '#0097A7','#D0F0F5'
        if d <  0.02: return '#3A5F7A','#D0E8F5'
        if d <  0.10: return '#C8920A','#FFF3CD'
        return '#C62828','#F8D7DA'

    if scenario_b and new_probs_b:
        st.markdown('<div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:8px;"><div style="font-family:Syne,sans-serif;font-size:13px;font-weight:700;color:#1565C0;padding:6px 10px;background:rgba(21,101,192,0.08);border-radius:6px;">📋 Scenario A</div><div style="font-family:Syne,sans-serif;font-size:13px;font-weight:700;color:#0097A7;padding:6px 10px;background:rgba(0,151,167,0.08);border-radius:6px;">📋 Scenario B</div></div>', unsafe_allow_html=True)
        sa_cols = st.columns(3)
        sb_cols = st.columns(3)
        for cols, new_probs, label_sfx in [(sa_cols, new_probs_a, 'A'), (sb_cols, new_probs_b, 'B')]:
            for col, lbl, bef, aft, icon in [
                (cols[0],'Sepsis Risk',   sep_base,  new_probs['sepsis'],   '🦠'),
                (cols[1],'Organ Failure', org_base,  new_probs['organ'],    '🫘'),
                (cols[2],'Mortality Risk',mort_base, new_probs['mortality'],'💀'),
            ]:
                diff = aft - bef
                dtext = f'{"▼" if diff<0 else "▲"} {abs(diff)*100:.1f}pp'
                fc, bc = _dc(bef, aft)
                with col:
                    st.markdown(
                        f'<div class="delta-card" style="background:{bc};border-color:{fc};">'
                        f'<div class="delta-label">{icon} {lbl} ({label_sfx})</div>'
                        f'<div class="delta-before">{bef*100:.1f}%</div>'
                        f'<div class="delta-after" style="color:{fc};">{aft*100:.1f}%</div>'
                        f'<div class="delta-change" style="color:{fc};">{dtext}</div></div>',
                        unsafe_allow_html=True
                    )
    else:
        dc1,dc2,dc3 = st.columns(3)
        for col, lbl, bef, aft, icon in [
            (dc1,'Sepsis Risk',   sep_base,  new_probs_a['sepsis'],   '🦠'),
            (dc2,'Organ Failure', org_base,  new_probs_a['organ'],    '🫘'),
            (dc3,'Mortality Risk',mort_base, new_probs_a['mortality'],'💀'),
        ]:
            diff = aft - bef
            dtext = f'{"▼" if diff<0 else "▲"} {abs(diff)*100:.1f}pp'
            fc, bc = _dc(bef, aft)
            with col:
                st.markdown(
                    f'<div class="delta-card" style="background:{bc};border-color:{fc};">'
                    f'<div class="delta-label">{icon} {lbl}</div>'
                    f'<div class="delta-before">{bef*100:.1f}%</div>'
                    f'<div class="delta-after" style="color:{fc};">{aft*100:.1f}%</div>'
                    f'<div class="delta-change" style="color:{fc};">{dtext}</div></div>',
                    unsafe_allow_html=True
                )

    # ── Comparison bar chart ──────────────────────────────────────────
    cats = ['Sepsis Risk','Organ Failure','Mortality']
    bv   = [sep_base*100, org_base*100, mort_base*100]
    av_a = [new_probs_a['sepsis']*100, new_probs_a['organ']*100, new_probs_a['mortality']*100]
    fig  = go.Figure()
    fig.add_trace(go.Bar(
        name='Baseline', x=cats, y=bv,
        marker=dict(color='#B5D8EC', line=dict(color='#6CA8CC',width=1)),
        text=[f'{v:.1f}%' for v in bv], textposition='outside',
        textfont=dict(family='JetBrains Mono',size=12,color='#3A5F7A'), width=0.25
    ))
    fig.add_trace(go.Bar(
        name='Scenario A', x=cats, y=av_a,
        marker=dict(color=['#1B8839' if a<b else '#C62828' for a,b in zip(av_a,bv)], line=dict(width=0)),
        text=[f'{v:.1f}%' for v in av_a], textposition='outside',
        textfont=dict(family='JetBrains Mono',size=12,color='#0D1B2A'), width=0.25
    ))
    if scenario_b and new_probs_b:
        av_b = [new_probs_b['sepsis']*100, new_probs_b['organ']*100, new_probs_b['mortality']*100]
        fig.add_trace(go.Bar(
            name='Scenario B', x=cats, y=av_b,
            marker=dict(color=['#0097A7' if a<b else '#D4500A' for a,b in zip(av_b,bv)],
                        line=dict(width=0), opacity=0.85),
            text=[f'{v:.1f}%' for v in av_b], textposition='outside',
            textfont=dict(family='JetBrains Mono',size=12,color='#0D1B2A'), width=0.25
        ))
    fig.update_layout(
        barmode='group', height=300,
        showlegend=True,
        legend=dict(font=dict(color='#1E3448',size=12), bgcolor='rgba(0,0,0,0)',
                    orientation='h', y=-0.25, xanchor='center', x=0.5),
        yaxis=dict(range=[0,115], title='Risk (%)', gridcolor='#90BDD8',
                   tickfont=dict(color='#0D1B2A',size=12)),
        xaxis=dict(tickfont=dict(color='#0D1B2A',size=14)),
        **PLOTLY_BASE
    )
    st.plotly_chart(fig, use_container_width=True)

    # ── P/F Ratio tracker ─────────────────────────────────────────────
    if ventilation != 'None':
        st.markdown('<div class="icu-section-label">Oxygenation — P/F Ratio Estimate</div>', unsafe_allow_html=True)
        pf_color = '#C62828' if pf_a < 100 else '#D4500A' if pf_a < 200 else '#C8920A' if pf_a < 300 else '#1B8839'
        pf_label = ('Severe ARDS (< 100)' if pf_a < 100 else
                    'Moderate ARDS (100–200)' if pf_a < 200 else
                    'Mild ARDS (200–300)' if pf_a < 300 else 'Normal (> 300)')
        pf1, pf2, pf3 = st.columns(3)
        with pf1:
            st.markdown(f'<div class="icu-kpi" style="--accent-color:{pf_color};"><div class="icu-kpi-label">P/F Ratio (estimated)</div><div class="icu-kpi-value" style="color:{pf_color};font-size:28px;">{pf_a:.0f}</div><div class="icu-kpi-sub">{pf_label}</div></div>', unsafe_allow_html=True)
        with pf2:
            st.markdown(f'<div class="icu-kpi" style="--accent-color:#0097A7;"><div class="icu-kpi-label">PEEP Applied</div><div class="icu-kpi-value" style="font-size:28px;">{peep} cmH₂O</div><div class="icu-kpi-sub">{"Lung-protective ✓" if peep >= 5 else "Consider higher PEEP"}</div></div>', unsafe_allow_html=True)
        with pf3:
            st.markdown(f'<div class="icu-kpi" style="--accent-color:#1565C0;"><div class="icu-kpi-label">FiO₂</div><div class="icu-kpi-value" style="font-size:28px;">{fio2}%</div><div class="icu-kpi-sub">{"⚠️ High FiO₂ — titrate down" if fio2 > 60 else "Acceptable range"}</div></div>', unsafe_allow_html=True)

    # ── Timeline ──────────────────────────────────────────────────────
    if timeline_a:
        tl_label = 'Intervention Timeline — Scenario A' if scenario_b else 'Intervention Timeline'
        st.markdown(f'<div class="icu-section-label">{tl_label}</div>', unsafe_allow_html=True)
        for t_time, t_color, t_text in timeline_a:
            st.markdown(
                f'<div class="timeline-item">'
                f'<div class="timeline-dot" style="background:{t_color};box-shadow:0 0 6px {t_color};"></div>'
                f'<div class="timeline-time">{t_time}</div>'
                f'<div class="timeline-text">{t_text}</div></div>',
                unsafe_allow_html=True
            )

    if scenario_b and timeline_b:
        st.markdown('<div class="icu-section-label">Intervention Timeline — Scenario B</div>', unsafe_allow_html=True)
        for t_time, t_color, t_text in timeline_b:
            st.markdown(
                f'<div class="timeline-item">'
                f'<div class="timeline-dot" style="background:{t_color};box-shadow:0 0 6px {t_color};"></div>'
                f'<div class="timeline-time">{t_time}</div>'
                f'<div class="timeline-text">{t_text}</div></div>',
                unsafe_allow_html=True
            )

    # ── Reassessment checkpoints ──────────────────────────────────────
    if any(reassess_a.values()):
        st.markdown('<div class="icu-section-label">Reassessment Checkpoints</div>', unsafe_allow_html=True)
        cp_colors = {'1h':'#C62828','3h':'#D4500A','6h':'#C8920A','24h':'#1565C0'}
        cp_cols   = st.columns(4)
        for col, (cp_time, cp_items) in zip(cp_cols, reassess_a.items()):
            if cp_items:
                items_html = ''.join(
                    f'<div style="font-family:Syne,sans-serif;font-size:12px;color:#1E3448;'
                    f'padding:5px 0;border-bottom:1px solid #B5D8EC;line-height:1.4;">✓ {item}</div>'
                    for item in cp_items
                )
                with col:
                    st.markdown(
                        f'<div style="background:#D0E8F5;border:1px solid #90BDD8;'
                        f'border-top:3px solid {cp_colors[cp_time]};border-radius:0 0 8px 8px;padding:12px 14px;">'
                        f'<div style="font-family:JetBrains Mono,monospace;font-size:11px;font-weight:700;'
                        f'color:{cp_colors[cp_time]};letter-spacing:0.12em;margin-bottom:10px;">@ {cp_time}</div>'
                        f'{items_html}</div>',
                        unsafe_allow_html=True
                    )

    # ── Summary banner ────────────────────────────────────────────────
    tmc = (new_probs_a['mortality'] - mort_base) * 100
    if tmc < -15: ac, msg = 'alert-ok',      f'Major mortality reduction of {abs(tmc):.1f}pp projected with current bundle.'
    elif tmc < -5: ac, msg = 'alert-ok',     f'Meaningful mortality reduction of {abs(tmc):.1f}pp projected.'
    elif tmc < 0:  ac, msg = 'alert-info',   f'Modest mortality improvement of {abs(tmc):.1f}pp projected.'
    elif tmc < 5:  ac, msg = 'alert-warning','Limited predicted benefit. Consider adding evidence-based bundle elements.'
    else:          ac, msg = 'alert-high',   f'⚠️ Selected interventions may increase mortality risk by {tmc:.1f}pp. Review contraindications.'

    st.markdown(
        f'<div class="icu-alert {ac}" style="margin-top:16px;padding:14px 18px;">'
        f'📊 <strong>Simulation Summary:</strong> {msg}</div>',
        unsafe_allow_html=True
    )

    # ── Handover export ───────────────────────────────────────────────
    st.markdown('<div class="icu-section-label">Handover Export</div>', unsafe_allow_html=True)
    sid      = int(patient.get('stay_id', 0))
    now_str  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    doc_id   = st.session_state.get('doctor_id','Unknown')

    lines = [
        f"ICU VIRTUAL SANDBOX — HANDOVER SUMMARY",
        f"Generated: {now_str}  |  Doctor: {doc_id}  |  Stay ID: {sid}",
        f"{'='*60}",
        f"",
        f"BASELINE RISKS:",
        f"  Sepsis:        {sep_base*100:.1f}%",
        f"  Organ Failure: {org_base*100:.1f}%",
        f"  Mortality:     {mort_base*100:.1f}%",
        f"  Criticality:   {tier}",
        f"",
        f"INTERVENTIONS — SCENARIO A:",
        f"  Antibiotics:   {abx_timing} ({abx_source})",
        f"  IV Fluids:     {fluids_ml} mL",
        f"  Vasopressor:   {vasopressor}" + (f" @ {vaso_dose:.2f} mcg/kg/min" if vasopressor != 'None' else ''),
        f"  Ventilation:   {ventilation}" + (f" | PEEP {peep} | FiO2 {fio2}%" if ventilation != 'None' else ''),
        f"  Prone:         {'Yes' if prone else 'No'}",
        f"  CRRT:          {'Yes' if dialysis else 'No'}",
        f"  Steroids:      {'Yes' if steroids else 'No'}",
        f"  Insulin:       {'Yes — target '+glucose_target if insulin else 'No'}",
        f"  Transfusion:   {'Yes — Hb '+str(hb_target)+' g/dL' if transfusion else 'No'}",
        f"  DVT Ppx:       {'Yes' if dvt_ppx else 'No'}",
        f"  Stress Ulcer:  {'Yes' if stress_ulcer else 'No'}",
        f"  Goals of Care: {'Yes' if goc else 'No'}",
        f"",
        f"PROJECTED OUTCOMES — SCENARIO A:",
        f"  Sepsis:        {new_probs_a['sepsis']*100:.1f}%  (Δ {(new_probs_a['sepsis']-sep_base)*100:+.1f}pp)",
        f"  Organ Failure: {new_probs_a['organ']*100:.1f}%  (Δ {(new_probs_a['organ']-org_base)*100:+.1f}pp)",
        f"  Mortality:     {new_probs_a['mortality']*100:.1f}%  (Δ {(new_probs_a['mortality']-mort_base)*100:+.1f}pp)",
        f"",
        f"REASSESSMENT CHECKLIST:",
    ]
    for cp_t, items in reassess_a.items():
        if items:
            lines.append(f"  @ {cp_t}:")
            for item in items:
                lines.append(f"    [ ] {item}")
    if warnings_a:
        lines.append(f"\nCLINICAL ALERTS:")
        for w_t, w_txt, _ in warnings_a:
            lines.append(f"  {w_t}: {w_txt}")
    lines += [
        f"",
        f"{'='*60}",
        f"⚠ Rule-based educational simulator. MIMIC-IV research use only.",
        f"  Not a validated clinical prediction tool.",
        f"  Always apply senior clinical judgment before acting.",
    ]
    handover_text = '\n'.join(lines)

    st.download_button(
        label='📋 Download Handover Summary (.txt)',
        data=handover_text,
        file_name=f'ICU_Sandbox_Stay{sid}_{datetime.date.today()}.txt',
        mime='text/plain',
        key='sandbox_export'
    )
    st.markdown(
        '<div style="font-family:JetBrains Mono,monospace;font-size:10px;color:#90BDD8;margin-top:8px;">'
        '⚠ Rule-based educational simulator · Not a validated clinical prediction tool · '
        'Always apply senior clinical judgment · MIMIC-IV research use only'
        '</div>',
        unsafe_allow_html=True
    )

# ══════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════
def main():
    if not st.session_state.get('logged_in', False):
        render_login()
        return

    col_profile, col_logout = st.columns([10, 1])
    with col_profile:
        doc_id  = st.session_state.get('doctor_id', '')
        now_str = __import__('datetime').datetime.now().strftime('%a %d %b %Y  %H:%M')
        st.markdown(
            f'''<div style="display:flex;align-items:center;gap:12px;padding:8px 0 4px 0;">
                <div style="width:38px;height:38px;border-radius:50%;
                            background:linear-gradient(135deg,#1565C0,#0097A7);
                            display:flex;align-items:center;justify-content:center;
                            font-size:18px;flex-shrink:0;">&#x1F468;&#x200D;&#x2695;&#xFE0F;</div>
                <div>
                    <div style="font-family:Syne,sans-serif;font-size:14px;font-weight:700;color:#0D1B2A;">
                        Dr. {doc_id}
                    </div>
                    <div style="font-family:JetBrains Mono,monospace;font-size:10px;color:#3A5F7A;
                                letter-spacing:0.06em;">{now_str} &nbsp;&middot;&nbsp; ICU AI Assistant</div>
                </div>
            </div>''',
            unsafe_allow_html=True)
    with col_logout:
        if st.button('🚪 Logout', key='logout_btn'):
            st.session_state['logged_in']   = False
            st.session_state['doctor_id']   = ''
            st.session_state['login_error'] = ''
            st.rerun()

    with st.spinner('Loading ICU data from Drive...'):
        try:
            (master, encoder, preds,
             fusion_meta, emb_meta,
             shap_sep, shap_mort, shap_organ,
             cxr_path_map) = load_data()
        except Exception as e:
            st.error(f'❌ Failed to load data: {e}')
            st.info('Ensure Google Drive is mounted and all parquet files exist.')
            return

    st.session_state['_master_ref'] = master
    render_hero(master)
    result = render_sidebar(master)
    if result[0] is None:
        st.warning('No patients found. Adjust filters.')
        return
    selected_sid, filtered_df, doctor_id = result
    if not doctor_id:
        doctor_id = st.session_state.get('doctor_id', '')

    patient_rows = master[master['stay_id'] == selected_sid]
    if len(patient_rows) == 0:
        st.error(f'Stay {selected_sid} not found.'); return
    patient = patient_rows.iloc[0]

    tab1, tab2, tab3, tab4, tab5, tab6, tab7, tab8, tab9 = st.tabs([
        '👤  Patient Overview',
        '📊  Clinical Scores',
        '🤖  AI Predictions',
        '🫁  Grad-CAM CXR',
        '📂  Modality Deep Dive',
        '📋  Patient History',
        '💊  Recommendations',
        '🔁  RLHF Feedback',
        '🧪  Sandbox',
    ])
    with tab1: tab_patient(patient, encoder)
    with tab2: tab_scores(patient, encoder)
    with tab3: tab_ai(patient, master, shap_sep, shap_mort, shap_organ, fusion_meta)
    with tab4: tab_gradcam(patient, cxr_path_map)
    with tab5: tab_modality(patient, encoder, fusion_meta, cxr_path_map)
    with tab6: tab_history(patient, master)
    with tab7: tab_recs(patient)
    with tab8: tab_feedback(patient, doctor_id)
    with tab9: tab_sandbox(patient)

if __name__ == '__main__':
    main()